In [ ]:
###Folders

# UNBLOCK-F6-01 (PATH): repoint at the writable mirror so the deposit is never written to (Figure5_Figure6 -- was a
#   Windows path)
Input_folder = '/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/replication_run/data/Intermediate_objects/'   # UNBLOCK-F6-01
# UNBLOCK-F6-01 (PATH): repoint + add the TRAILING SLASH: every save is string concatenation (Output_folder+'x.svg'), so
#   without it figures land as a sibling file
Output_folder = '/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/replication_run/Output_folder/ephys/'   # UNBLOCK-F6-01


'Download data from:  https://doi.org/10.17605/OSF.IO/3D9R2'

In [ ]:
##Importing libraries
from joblib import dump, load
import os, sys, pickle, time, re, csv
from collections import defaultdict#

import numpy as np
import pandas as pd

import scipy.stats as st
import math

import matplotlib.pyplot as plt
import seaborn as sns
import collections, numpy

from itertools import groupby
from pingouin import partial_corr
from collections import Counter
import random
from sklearn.linear_model import LogisticRegression
from scipy.stats import circmean
from scipy.ndimage import gaussian_filter1d
import warnings
import statsmodels
from sklearn.preprocessing import MaxAbsScaler
from scipy import stats

In [ ]:
from collections import defaultdict
def rec_dd():
    return defaultdict(rec_dd)

def remove_empty(xx):

    yy= [x for x in xx if len(x) > 0]
    return(yy)

def rand_jitterX(arr, X):
    stdev = X*(max(arr)-min(arr))
    return arr + np.random.randn(len(arr)) * stdev

##convert nested dict into array
def dict_to_array(d):
    dictlist=[]
    for key, value in d.items():
        dictlist.append(value)
    return(np.asarray(dictlist))

def rank_repeat(a):
    arr=np.zeros(len(a))
    for n in np.unique(a):
        count=0
        for ii in range(len(a)):
            if a[ii]==n:
                arr[ii]=count
                count+=1

    arr=arr.astype(int)
    return(arr)

def concatenate_complex2(xx):

    ALL_elements=[]
    for ii in np.arange(len(xx)):
        xxii=xx[ii]
        for jj in np.arange(len(xxii)):
            xxiijj=xxii[jj]
            ALL_elements.append(np.asarray(xxiijj))
            
    return(np.asarray(ALL_elements))

def smooth_circular(x,sigma=10):
    return(gaussian_filter1d(np.hstack((x,x,x)),sigma,axis=0)[len(x):int(len(x)*2)])

def polar_plot_stateX2(meanx,upperx,lowerx,ax,repeated,color='black',labels='states',plot_type='line',Marker=False,\
                      fields_booleanx=[], structure_abstract='ABCD',fontsize=20,set_max=False,max_val=1):
    rx = list(meanx)
    theta = list(range(len(rx)))
    thetax = [2 * np.pi * (x/len(rx)) for x in theta]
    r = rx + [rx[0]]
    theta = thetax + [thetax[0]]
    
    #ax=plt.subplot(111, projection='polar')
    
    if Marker==True:
        fields_booleanx=fields_booleanx*(np.max(upperx)+0.1*np.max(upperx))
        fields_boolean=list(fields_booleanx)+[list(fields_booleanx)[0]]

    upper=list(upperx)+[list(upperx)[0]]
    lower=list(lowerx)+[list(lowerx)[0]]
    
    if plot_type=='line':
        ax.plot(theta, r,color=color)
        ax.fill_between(theta, upper, lower, alpha=0.2,color=color)
        if set_max==False:
            ax.set_rmax(np.max(upper)+0.01*np.max(upper))
        else:
            ax.set_rmax(max_val)
            
        if Marker==True:
            ax.plot(theta, fields_boolean,color='black',linestyle='None',marker='.')

    elif plot_type=='bar':
        ax.bar(theta,r,width=5/len(r),color=color)
    elif plot_type=='marker':
        ax.plot(theta, r,color=color)
        
    
    ax.grid(True)
    #ax.set_rorigin(-1)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    if labels=='states':
        if structure_abstract=='ABCD':
            ax.set_xticklabels(['A', '', 'B', '', 'C', '', 'D', ''],fontsize=fontsize)
        elif structure_abstract=='AB' and repeated==False:
            ax.set_xticklabels(['A', '', '', '', 'B', '', '', ''],fontsize=fontsize)
        elif structure_abstract=='AB' and repeated==True:
            ax.set_xticklabels(['A', '', 'B', '', 'A', '', 'B', ''],fontsize=fontsize)
    elif labels == 'angles':
        ax.set_xticklabels(['0', '', '90', '', '180', '', '270', ''],fontsize=fontsize)

def indep_roll(arr, shifts, axis=1):
    """Apply an independent roll for each dimensions of a single axis.

    Parameters
    ----------
    arr : np.ndarray
        Array of any shape.

    shifts : np.ndarray
        How many shifting to use for each dimension. Shape: `(arr.shape[axis],)`.

    axis : int
        Axis along which elements are shifted. 
    """
    arr = np.swapaxes(arr,axis,-1)
    all_idcs = np.ogrid[[slice(0,n) for n in arr.shape]]

    # Convert to a positive shift
    shifts[shifts < 0] += arr.shape[-1] 
    all_idcs[-1] = all_idcs[-1] - shifts[:, np.newaxis]

    result = arr[tuple(all_idcs)]
    arr = np.swapaxes(result,-1,axis)
    return arr 

def bar_plotX(y,name,ymin,ymax,points,pairing,jitt):
    leny=len(y)
    plt.figure(figsize=(leny*(3/2),6))
    
    if ymin =='auto':
        ymin=np.min(np.concatenate(y))
    if ymax =='auto':
        ymax=np.max(np.concatenate(y))
    
    ##bars
    y_mean=((np.zeros(len(y))))
    y_sem=((np.zeros(len(y))))
    for ii in range(0, len(y)):
        ymeanx=np.nanmean(y[ii])
        y_mean[ii]=ymeanx
        ysemx=st.sem(y[ii], nan_policy='omit')
        y_sem[ii]=ysemx
   
    
    xxx=np.linspace(0.15, 0.2+(0.2*(leny-1)), leny)

    xlocations = np.array(range(len(xxx)))
    width=0.2
    plt.bar(xxx, y_mean, width, yerr=y_sem, alpha=1, 
           error_kw=dict(ecolor='gray', lw=2, capsize=5, capthick=2), align='center')
    
    if points != 'points' and ymin == 'auto':
        ymin=np.min(y_mean-y_sem) #-np.max(y_sem)
        ymax=np.max(y_mean+y_sem) #+np.max(y_sem)
    
    #if ymin>0:
    #    ymin=0
    plt.ylim(ymin-(0.05*(ymax-ymin)),ymax+(0.05*(ymax-ymin)))
    plt.xlim(0,np.max(xxx)+0.15)
    
    

    ###points and lines
    if points == 'points':
        yyALL=[]
        for ii in range(0, len(y)):
            yy=np.column_stack((y[ii],np.repeat(xxx[ii],len(y[ii]))))
            yyALL.append(yy)

        xy=np.vstack((yyALL))
        jittered=rand_jitterX(xy[:,1],jitt)

        if pairing == 'paired':
            for ii in range(0, leny):
                x1=np.split(jittered,len(y))[ii]
                if ii == 0:
                    x1_all=x1
                else:
                    x1_all=np.column_stack((x1_all,x1))

            for jj in range(0,np.shape(y)[1]):
                yyyy=np.asarray(y)[:,jj]
                plt.plot(x1_all[jj],yyyy, color='gray')
        plt.plot(jittered,xy[:,0],'o',markersize=7,color='white',markeredgecolor='black')
    
    if name != 'none':
        plt.savefig(name)
    
    #plt.show()

###function to plot scatter plots (e.g. comparing assembly strength at correct vs incorrect dispensers)
def plot_scatter(x,y,name='none'):
    plt.plot(x, y, 'o')
    z= [-10000, 0, 10000]
    plt.plot(z,z,'k--')

    xy=np.hstack((x,y))

    xmin=min(xy)-np.mean(xy)*0.1
    xmax=max(xy)+np.mean(xy)*0.1
    ymin=min(xy)-np.mean(xy)*0.1
    ymax=max(xy)+np.mean(xy)*0.1

    plt.xlim(xmin,xmax)
    plt.ylim(ymin,ymax)
    
    plt.gca().set_aspect('equal', adjustable='box')
    
    if name != 'none':
        plt.savefig(name)
    plt.show()
    

    
def circular_sem(a):
    if len(np.shape(a))==2:
        sem_=np.rad2deg(np.hstack(([st.circvar(remove_nan(a[:,ii]))/np.sqrt(len(remove_nan(a[:,ii])))\
                               for ii in range(len(a.T))])))
    elif len(np.shape(a))==1:
        sem_=np.rad2deg(st.circvar(remove_nan(a))/np.sqrt(len(remove_nan(a))))
        
    return(sem_)
    
def non_repeat_ses_maker(mouse_recday):
    Tasks=np.load(Input_folder+'Task_data_'+mouse_recday+'.npy',allow_pickle=True)
    num_trials_day=np.load(Input_folder+'Num_trials_'+mouse_recday+'.npy',\
                                        allow_pickle=True)

    non_repeat_bool_all=[]
    for ses_ind in np.arange(len(Tasks)):
        if ses_ind==0:
            non_repeat_bool=True
        else:
            num_prev_repeats=np.sum([np.array_equal(Tasks[ses_ind],Tasks[:ses_ind][jj])\
                                     for jj in range(len(Tasks[:ses_ind]))])
            if num_prev_repeats==0:
                non_repeat_bool=True
            else:
                non_repeat_bool=False

        non_repeat_bool_all.append(non_repeat_bool)
    non_repeat_bool_all=np.hstack((non_repeat_bool_all))
    num_trials_bool=num_trials_day>0
    non_repeat_ses_bool=np.logical_and(non_repeat_bool_all,num_trials_bool)

    non_repeat_ses=np.where(non_repeat_ses_bool==True)[0]
    return(non_repeat_ses)

def two_proportions_test(success_a, size_a, success_b, size_b):
    """
    A/B test for two proportions;
    given a success a trial size of group A and B compute
    its zscore and pvalue
    
    Parameters
    ----------
    success_a, success_b : int
        Number of successes in each group
        
    size_a, size_b : int
        Size, or number of observations in each group
    
    Returns
    -------
    zscore : float
        test statistic for the two proportion z-test

    pvalue : float
        p-value for the two proportion z-test
    """
    prop_a = success_a / size_a
    prop_b = success_b / size_b
    prop_pooled = (success_a + success_b) / (size_a + size_b)
    var = prop_pooled * (1 - prop_pooled) * (1 / size_a + 1 / size_b)
    zscore = np.abs(prop_b - prop_a) / np.sqrt(var)
    one_side = 1 - stats.norm(loc = 0, scale = 1).cdf(zscore)
    pvalue = one_side * 2
    return zscore, pvalue

def partition(alist, indices):
    return np.asarray([np.asarray(alist[i:j]) for i, j in zip(indices[:-1], indices[1:])])

def normalise(xx,num_bins=90,take_max=False):
    lenxx=len(xx)
    if lenxx<num_bins:
        xx=np.repeat(xx,10)/10
        lenxx=lenxx*10
    indices_polar=np.arange(lenxx)
    if take_max==True:
        normalized_xx=st.binned_statistic(indices_polar,xx, 'max', bins=num_bins)[0]
    else:
        normalized_xx=st.binned_statistic(indices_polar,xx, 'mean', bins=num_bins)[0]
    return(normalized_xx)

def raw_to_norm(raw_neuron,Trial_times_conc,num_states=4,return_mean=True,smoothing=True,\
                take_max=False,smoothing_sigma=10):
    raw_neuron_split=remove_empty(partition(list(raw_neuron),list(Trial_times_conc)))
    if len(raw_neuron_split)%num_states!=0:
        raw_neuron_split=raw_neuron_split[:len(raw_neuron_split)-len(raw_neuron_split)%num_states]
    
    if take_max==True:
        raw_neuron_split_norm=np.asarray([normalise(raw_neuron_split[ii],take_max=True)\
                                          for ii in np.arange(len(raw_neuron_split))])
    else:
        raw_neuron_split_norm=np.asarray([normalise(raw_neuron_split[ii]) for ii in np.arange(len(raw_neuron_split))])
    
    Actual_norm=(raw_neuron_split_norm.reshape(len(raw_neuron_split_norm)//num_states,\
                                               len(raw_neuron_split_norm[0])*num_states))
    
    if return_mean==True:
        Actual_norm_mean=np.nanmean(Actual_norm,axis=0)
        if smoothing==True:
            Actual_norm_smoothed=smooth_circular(Actual_norm_mean,sigma=smoothing_sigma)
            return(Actual_norm_smoothed)
        else:
            return(Actual_norm_mean)
    else:
        return(Actual_norm)
    
def remove_nan(x):
    x=x[~np.isnan(x)]
    return(x)

def unique_nosort(a):
    indexes = np.unique(a, return_index=True)[1]
    return(np.asarray([a[index] for index in sorted(indexes)]))

def one_hot_encode(x,length):
    array=np.zeros((len(x),length))
    for entry in np.arange(len(x)):
        if ~np.isnan(x[entry]):
            array[entry,int(x[entry])]=1
    return(array)

def data_matrix(data, concatenate=False):
    data_mat=np.asarray([data[ii][:len(data[-1])] for ii in range (len(data))])
    if concatenate==True:
        data_mat=np.concatenate(np.hstack(data_mat))
    return(data_mat)

###counts num of repeats for each stretch of numbers
def rank_repeat2(a):
    num_repeats=number_of_repeats(a)
    arr=[]
    for n_ind, n in enumerate(unique_adjacent(a)):
        count=0
        indices=np.arange(num_repeats[n_ind])
        arr.append(indices)
    arr=np.concatenate(arr)
    arr=arr.astype(int)
    return(arr)

def number_of_repeats(array):
    return(np.asarray([sum(1 for _ in group) for _, group in groupby(array)]))

def unique_adjacent(a):
    return(np.asarray([k for k,g in groupby(a)]))

def matrix_triangle(a,direction='upper',return_indices=False):
    if direction=='upper':
        indices=np.triu_indices(len(a), k = 1)
    if direction=='lower':
        indices=np.tril_indices(len(a), k = -1)
    triangle=a[indices]
    if return_indices==True:
        return(triangle,indices)
    else:
        return(triangle)
    
    
def edge_node_fill(edge_mat,node_mat):
    new_mat=np.copy(edge_mat)
    for ii in [0,2,4]:
        new_mat[ii,0]=node_mat[int(ii/2),0]
        new_mat[ii,2]=node_mat[int(ii/2),1]
        new_mat[ii,4]=node_mat[int(ii/2),2]
        
    return(new_mat)

def _nanargmin(arr, axis=0):
    try:
        if len(np.shape(arr))==1:
            return np.nanargmin(arr)
        else:
            return np.nanargmin(arr, axis)
    except ValueError:
        return np.nan
    
def _nanargmax(arr, axis=0):
    try:
        if len(np.shape(arr))==1:
            return np.nanargmax(arr)
        else:
            return np.nanargmax(arr, axis)
    except ValueError:
        return np.nan

from collections import Counter
from itertools import combinations
def most_common_pair(a_):
    a=np.copy(a_)
    d  = Counter()
    for sub in a:
        if len(a) < 2:
            continue
        #sub.sort()
        for comb in combinations(sub,2):
            d[comb] += 1

    return([d.most_common()[0][0][0],d.most_common()[0][0][1]], d.most_common()[0][1]/len(a))

def polar_plot_stateX(meanx,upperx,lowerx,color='black',labels='states',plot_type='line',Marker=False,\
                      fields_booleanx=None):
    rx = list(meanx)
    theta = list(range(len(rx)))
    thetax = [2 * np.pi * (x/len(rx)) for x in theta]
    r = rx + [rx[0]]
    theta = thetax + [thetax[0]]
    
    if Marker==True:
        fields_booleanx=fields_booleanx*(np.max(upperx)+0.1*np.max(upperx))
        fields_boolean=list(fields_booleanx)+[list(fields_booleanx)[0]]

    upper=list(upperx)+[list(upperx)[0]]
    lower=list(lowerx)+[list(lowerx)[0]]
    
    ax = plt.subplot(111, projection='polar')
    
    if plot_type=='line':
        ax.plot(theta, r,color=color)
        ax.fill_between(theta, upper, lower, alpha=0.2,color=color)
        ax.set_rmax(np.max(upper)+0.01*np.max(upper))
        if Marker==True:
            ax.plot(theta, fields_boolean,color='black',linestyle='None',marker='.')

    elif plot_type=='bar':
        ax.bar(theta,r,width=5/len(r),color=color)
    elif plot_type=='marker':
        ax.plot(theta, r,color=color)
        
    
    ax.grid(True)
    #ax.set_rorigin(-1)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    if labels=='states':
        ax.set_xticklabels(['A', '', 'B', '', 'C', '', 'D', ''],fontsize=20)
    elif labels == 'angles':
        ax.set_xticklabels(['0', '', '90', '', '180', '', '270', ''],fontsize=20)

    #plt.show()
    
def fill_diagonal(source_array, diagonal):
    copy = source_array.copy()
    np.fill_diagonal(copy, diagonal)
    return copy

def column_stack_clean(x,y):
    xy=np.column_stack((x,y))
    xy=xy[~np.isnan(xy).any(axis=1)]
    xy=xy[~np.isinf(xy).any(axis=1)]
    x=xy[:,0]
    y=xy[:,1]
    xy_new=np.column_stack((x,y))
    return(xy_new)

def noplot_scatter(x,y, color):
    plt.plot(x, y, 'o', color=color, alpha=0.7,markersize=7)
    z= [-10000, 0, 10000]
    plt.plot(z,z,'k--')

    xy=np.hstack((x,y))
    
    global xmin
    global xmax
    global ymin
    global ymax
    
    xmin=min(xy)-np.mean(xy)*0.1
    xmax=max(xy)+np.mean(xy)*0.1    
    ymin=min(xy)-np.mean(xy)*0.1
    ymax=max(xy)+np.mean(xy)*0.1

    


    #plt.xlim(-0.2,0.2)
    #plt.ylim(-0.2,0.2)

    plt.xlim(xmin,xmax)
    plt.ylim(ymin,ymax)
    plt.gca().set_aspect('equal', adjustable='box')
    

In [ ]:
#LOADING FILES - tracking, behaviour, Ephys raw, Ephys binned
tt=time.time()


try:
    os.mkdir(Input_folder)
except FileExistsError:
    pass

dictionaries_list=['Xneuron_correlations']

##,'binned_FR_dic'

for name in dictionaries_list:
    try:
        data_filename_memmap = os.path.join(Input_folder, name)
        data = load(data_filename_memmap)#, mmap_mode='r')
        exec(name+'= data')
    except Exception as e:
        print(name)
        print(e)
        print('Not loaded')
print(time.time()-tt)


for day_type in Xneuron_correlations.keys():
    for measure in Xneuron_correlations[day_type].keys():
        for mouse_recday in Xneuron_correlations[day_type][measure].keys():
            np.save(Input_folder+'Xneuron_correlations_'+day_type+'_'+measure+'_'+mouse_recday+'.npy',\
                    Xneuron_correlations[day_type][measure][mouse_recday])
            
        


In [ ]:
##Defining Task grid
from scipy.spatial import distance_matrix
from itertools import product
len_side=3
x=np.arange(len_side)
S=np.asarray(list(product(x, x)))
Task_matrix_blank=np.zeros((len_side,len_side))

A = [[-1, 0], [1, 0], [0, 1], [0, -1]]

###shortest distances 
from scipy.spatial import distance_matrix
from itertools import product
x=(0,1,2)
Task_grid=np.asarray(list(product(x, x)))

Task_grid_plotting=np.column_stack((Task_grid[:,1],Task_grid[:,0]))
Task_grid_plotting2=[]
for yy in np.arange(3):
    y=int(yy*2)
    for xx in np.arange(3):
        x=int(xx*2)    
        Task_grid_plotting2.append([x,y])
Task_grid_plotting2=np.asarray(Task_grid_plotting2)    
Task_grid2=np.column_stack((Task_grid_plotting2[:,1],Task_grid_plotting2[:,0]))

Edge_grid=np.asarray([[1,2],[2,3],[1,4],[2,5],[3,6],[4,5],[5,6],[4,7],[5,8],[6,9],[7,8],[8,9]]) ###
Edge_grid_=Edge_grid-1
Edge_grid_coord_x=[Task_grid[Edge_grid_[ii][0]][0]+Task_grid[Edge_grid_[ii][1]][0] for ii in range(len(Edge_grid_))]
Edge_grid_coord_y=rank_repeat(Edge_grid_coord_x)
Edge_grid_coord=np.column_stack((Edge_grid_coord_x,Edge_grid_coord_y))
Edge_grid_coord2=np.asarray([[0,1],[0,3],[1,0],[1,2],[1,4],[2,1],[2,3],[3,0],[3,2],[3,4],[4,1],[4,3]])

direction_dic={'N':[1,0],'S':[-1,0],'W':[0,1],'E':[0,-1]}
direction_dic_plotting={'N': [0, -1], 'S': [0, 1], 'W': [-1, 0], 'E': [1, 0]}

mapping_pyth={2:2,5:3,8:4}

distance_mat_raw=distance_matrix(Task_grid, Task_grid)
len_matrix=len(distance_mat_raw)
distance_mat=np.zeros((len_matrix,len_matrix))
for ii in range(len_matrix):
    for jj in range(len_matrix):
        if (distance_mat_raw[ii,jj]).is_integer()==False:
            hyp=int((distance_mat_raw[ii,jj])**2)
            distance_mat[ii,jj]=mapping_pyth[hyp]
        else:
            distance_mat[ii,jj]=distance_mat_raw[ii,jj]
mindistance_mat=distance_mat.astype(int)

In [ ]:
###Is spatial anchoring conserved across tasks?

tt=time.time()

Spatial_anchoring_dic=rec_dd()

day_type='combined_ABCDonly'
thr_visit=2
num_phases=3
num_locations=9
num_locations_withedges=21
num_states=4
len_phase=30
num_lags=num_states*num_phases
num_bins_total=num_lags*len_phase


use_prefphase=True
smoothing_sigma=10
lag_min=90 ##for circular shifts
skip_analysed=True
run_perm=False
num_iterations=100


for mouse_recday in np.load(Input_folder+day_type+'_days.npy'):
    print(mouse_recday)
    num_sessions=len(np.load(Input_folder+'awake_session_behaviour_'+mouse_recday+'.npy'))
    num_neurons=len(np.load(Input_folder+'Neuron_raw_'+mouse_recday+'_0.npy'))

    if skip_analysed==True:
        if len(Spatial_anchoring_dic['rotation_angle_mat'][mouse_recday])==num_neurons:
            print('Already analyzed')
            continue
    
    #Importing Ephys
    print('Importing Ephys')
    num_sessions=len(np.load(Input_folder+'awake_session_behaviour_'+mouse_recday+'.npy'))
    ses_not_found=[]
    for session in np.arange(num_sessions):
        try:
            ephys_ = np.load(Input_folder+'Neuron_'+mouse_recday+'_'+str(session)+'.npy')
            exec('ephys_ses_'+str(session)+'_=ephys_')
        except:
            print('Ephys not found')
            ses_not_found.append(session)


    ##Calculating rotations
    print('Calculating rotations')
    rotation_dist_mat=np.zeros((num_neurons,num_phases,num_locations,num_sessions,num_sessions))
    rotation_angle_mat=np.zeros((num_neurons,num_phases,num_locations,num_sessions,num_sessions))
    mean_rotation_dist_all=np.zeros((num_neurons,num_phases,num_locations))
    num_passes_all=np.zeros((num_sessions,num_phases,num_locations))
    best_node_phase=np.zeros((num_neurons,2))

    mean_FRs_all=np.zeros((num_neurons,num_sessions,num_lags,num_locations_withedges))
    mean_FRs_shuff_all=np.zeros((num_neurons,num_sessions,num_iterations,num_locations_withedges))

    rotation_dist_mat[:]=np.nan
    rotation_angle_mat[:]=np.nan
    mean_rotation_dist_all[:]=np.nan
    num_passes_all[:]=np.nan
    best_node_phase[:]=np.nan
    mean_FRs_all[:]=np.nan
    mean_FRs_shuff_all[:]=np.nan

    for neuron in np.arange(num_neurons):
        print(neuron)
        for session in np.arange(num_sessions):
            if session in ses_not_found:
                continue
            exec('ephys_=ephys_ses_'+str(session)+'_')

            location_mat_=np.load(Input_folder+'Location_'+mouse_recday+'_'+str(session)+'.npy')
            if len(location_mat_)==0:
                continue
            occupancy_mat=np.reshape(location_mat_,(num_states,len(location_mat_),len(location_mat_.T)//num_states))


            ephys_neuron_=ephys_[neuron]
            neuron_mat=ephys_neuron_

            phase_mat=np.zeros(np.shape(occupancy_mat))
            phase_mat[:,:,30:60]=1
            phase_mat[:,:,60:90]=2
            phase_conc=np.concatenate(np.hstack(phase_mat))

            phase_peaks=np.load(Input_folder+'tuning_phase_boolean_max_'+mouse_recday+'.npy')[session]

            pref_phase_neurons=np.argmax(phase_peaks,axis=1)
            pref_phase=pref_phase_neurons[neuron]

            mean_FRs_neuronsession=np.zeros((num_lags,num_locations_withedges))
            mean_FRs_neuronsession[:]=np.nan
            mean_FRs_shuff_session=np.zeros((num_iterations,num_locations_withedges))
            mean_FRs_shuff_session[:]=np.nan

            if len(neuron_mat)==0 or len(occupancy_mat)==0:
                if neuron==0 and session==0:
                    print('No data for session'+str(session))

            else:
                occupancy_conc=np.concatenate(location_mat_)
                neuron_conc=np.concatenate(neuron_mat)

                min_len=np.min([len(occupancy_conc),len(neuron_conc)])
                neuron_conc=neuron_conc[:min_len]
                occupancy_conc=occupancy_conc[:min_len]
                phase_conc=phase_conc[:min_len]

                ##Shifting
                for shift in np.arange(num_lags):
                    occupancy_shifted=np.roll(occupancy_conc,len_phase*shift) 
                    ###i.e. if peak in 4th bin (bin index 3) then anchor is 4 bins behind peak of activity 
                    occupancy_shifted_nonan=occupancy_shifted[~np.isnan(occupancy_shifted)]
                    neuron_conc_nonan=neuron_conc[~np.isnan(occupancy_shifted)]

                    occupancy_shifted_nonan_prefphase=occupancy_shifted[np.logical_and(~np.isnan(occupancy_shifted),\
                                                                           phase_conc==pref_phase)]
                    neuron_conc_nonan_pref_phase=neuron_conc[np.logical_and(~np.isnan(occupancy_shifted),\
                                                                            phase_conc==pref_phase)]

                    if use_prefphase==True:
                        mean_FRs=st.binned_statistic(occupancy_shifted_nonan_prefphase,\
                                        neuron_conc_nonan_pref_phase,bins=np.arange(22)+1,statistic='mean')[0]
                    else:
                        mean_FRs=st.binned_statistic(occupancy_shifted_nonan, neuron_conc_nonan,bins=\
                                                 np.arange(num_locations_withedges+1)+1,statistic='mean')[0]

                    mean_FRs_neuronsession[shift]=mean_FRs

                ###Permutations (random circular shift)
                if run_perm==True:
                    occupancy_conc_shuff=np.split(np.copy(occupancy_conc),len(occupancy_conc)/90)
                    max_roll=int(len(occupancy_conc)-lag_min)
                    min_roll=int(lag_min)

                    for iteration in np.arange(num_iterations):
                        shift_rand=random.randrange(max_roll-min_roll)+min_roll
                        occupancy_conc_shuff=np.hstack((occupancy_conc_shuff))
                        occupancy_conc_shuff=np.roll(occupancy_conc_shuff,shift_rand)
                        occupancy_conc_shuff_nonan=occupancy_conc_shuff[~np.isnan(occupancy_conc_shuff)]
                        neuron_conc_nonan=neuron_conc[~np.isnan(occupancy_conc_shuff)]

                        occupancy_shuff_nonan_prefphase=occupancy_conc_shuff[np.logical_and\
                                                                             (~np.isnan(occupancy_conc_shuff),\
                                                                                           phase_conc==pref_phase)]
                        neuron_conc_nonan_pref_phase=neuron_conc[np.logical_and(~np.isnan(occupancy_conc_shuff),\
                                                                                phase_conc==pref_phase)]

                        if use_prefphase==True:
                            mean_FRs_shuff=st.binned_statistic(occupancy_shuff_nonan_prefphase,\
                                                               neuron_conc_nonan_pref_phase,\
                                                bins=np.arange(num_locations_withedges+1)+1,statistic='mean')[0]
                        else:
                            mean_FRs_shuff=st.binned_statistic(occupancy_conc_shuff_nonan, neuron_conc_nonan,\
                                                bins=np.arange(num_locations_withedges+1)+1,statistic='mean')[0]


                        mean_FRs_shuff_session[iteration]=mean_FRs_shuff


            mean_FRs_all[neuron][session]=mean_FRs_neuronsession
            if run_perm==True:
                mean_FRs_shuff_all[neuron][session]=mean_FRs_shuff_session



        for phase_ in np.arange(num_phases):
            for location_ in np.arange(num_locations)+1:
                mean_smooth_all=np.zeros((num_sessions,num_bins_total))
                for session in np.arange(num_sessions):
                    exec('ephys_=ephys_ses_'+str(session)+'_')
                    try:
                        location_mat_=np.load(Input_folder+'Location_'+mouse_recday+'_'+str(session)+'.npy')
                    except:
                        print('Location file not found')
                        continue
                    if len(location_mat_)==0:
                        continue
                    occupancy_mat=np.reshape(location_mat_,\
                                             (num_states,len(location_mat_),len(location_mat_.T)//num_states))

                    ephys_neuron_=ephys_[neuron]
                    neuron_mat=data_matrix(ephys_neuron_,concatenate=False)

                    if len(neuron_mat)==0 or len(occupancy_mat)==0:
                        if neuron==0 and location_==1 and phase_==0:
                            print('No data for session'+str(session))
                        mean_smooth_all[session]=np.repeat(np.nan,num_bins_total)
                        num_passes_all[session,phase_,int(location_-1)]=0
                        continue

                    occupancy_conc=np.concatenate(location_mat_)
                    phase_mat=np.zeros(np.shape(occupancy_mat))
                    phase_mat[:,:,30:60]=1
                    phase_mat[:,:,60:90]=2
                    phase_conc=np.concatenate(np.hstack(phase_mat))
                    neuron_conc=np.concatenate(neuron_mat)



                    timestamps_=np.where((np.logical_and(occupancy_conc==location_, phase_conc==phase_)))[0]
                    long_stays=np.where(rank_repeat2(occupancy_conc)>thr_visit-1)[0]
                    timestamps=np.intersect1d(timestamps_,long_stays-(thr_visit+1))

                    if len(timestamps)>0:

                        timestamps_start=timestamps[(np.hstack((1,np.diff(timestamps)>thr_visit))).astype(bool)]
                        timestamps_end=timestamps[(np.hstack((np.diff(timestamps)>thr_visit,1))).astype(bool)]
                        time_stamps_used=timestamps_start

                        aligned_activity=np.asarray([neuron_conc[ii:ii+num_bins_total]\
                                                     if len(neuron_conc[ii:])>=num_bins_total else\
                                                     np.repeat(np.nan,num_bins_total) for ii in time_stamps_used])


                        mean_=np.nanmean(aligned_activity,axis=0)
                        sem_=st.sem(aligned_activity,axis=0,nan_policy='omit')
                        mean_smooth=smooth_circular(mean_,sigma=smoothing_sigma)
                        sem_smooth=smooth_circular(sem_,sigma=smoothing_sigma)

                        if np.nanmean(mean_smooth)==0 or np.isnan(np.nanmean(mean_smooth))==True:
                            mean_smooth=np.repeat(np.nan,num_bins_total)

                        mean_smooth_all[session]=mean_smooth

                        num_passes_all[session,phase_,int(location_-1)]=len(time_stamps_used)

                    else:
                        mean_smooth_all[session]=np.repeat(np.nan,num_bins_total)
                        num_passes_all[session,phase_,int(location_-1)]=0

                for indX in np.arange(num_sessions):
                    for indY in np.arange(num_sessions):
                        if indX!=indY:
                            if np.logical_and(~np.isnan(np.nanmean(mean_smooth_all[indX])),\
                                                                        ~np.isnan(np.nanmean(mean_smooth_all[indY]))):
                                shifted_corrs=np.asarray([st.pearsonr(mean_smooth_all[indX],\
                                                                      np.roll(mean_smooth_all[indY],int(n*10)))[0]\
                                                          for n in np.arange(num_bins_total/10) ])
                                rotation=np.argmax(shifted_corrs)*10
                                rotation_dist=1-math.cos(np.deg2rad(rotation))
                            else:
                                rotation=np.nan
                                rotation_dist=np.nan



                        else:
                            rotation=0
                            rotation_dist=0

                        rotation_dist_mat[neuron,phase_,int(location_-1),indX,indY]=rotation_dist
                        rotation_angle_mat[neuron,phase_,int(location_-1),indX,indY]=rotation

                mean_rotation_dist=np.nanmean(matrix_triangle(rotation_dist_mat[neuron,phase_,int(location_-1)],\
                                                              direction='lower'))
                mean_rotation_dist_all[neuron,phase_,int(location_-1)]=mean_rotation_dist

                if np.min(num_passes_all[:,phase_,int(location_-1)])<=thr_visit:
                    mean_rotation_dist_all[neuron,phase_,int(location_-1)]=np.nan



        best_anchor_=np.where(mean_rotation_dist_all[neuron]==np.nanmin(mean_rotation_dist_all[neuron]))
        if len(best_anchor_[0])==1:
            best_anchor_phase=best_anchor_[0][0]
            best_anchor_node=best_anchor_[1][0]+1
        elif len(best_anchor_[0])>1:
            print('Multiple best anchors')
        elif len(best_anchor_[0])==0:
            best_anchor_node=np.nan
            best_anchor_phase=np.nan

        best_node_phase[neuron]=np.asarray([best_anchor_node,best_anchor_phase])


    Spatial_anchoring_dic['rotation_angle_mat'][mouse_recday]=rotation_angle_mat
    Spatial_anchoring_dic['rotation_dist_mat'][mouse_recday]=rotation_dist_mat
    Spatial_anchoring_dic['mean_rotation_dist_all'][mouse_recday]=mean_rotation_dist_all
    Spatial_anchoring_dic['num_passes_all'][mouse_recday]=num_passes_all
    Spatial_anchoring_dic['best_node_phase'][mouse_recday]=best_node_phase

    if run_perm==True:
        Spatial_anchoring_dic['Shuffled_maps'][mouse_recday]=mean_FRs_shuff_all

print(time.time()-tt)

In [ ]:
###Running just the lagged spatial map analysis

thr_visit=2
num_phases=3

num_locations=9
num_locations_withedges=21
num_lags=12
len_phase=int(360/num_lags)
num_iterations=100

use_prefphase=True

lag_min=90 ##for circular shifts

for mouse_recday in np.load(Input_folder+'combined_ABCDonly_days.npy'):
    print(mouse_recday)
    num_sessions=len(np.load(Input_folder+'awake_session_behaviour_'+mouse_recday+'.npy'))
    num_neurons=len(np.load(Input_folder+'Neuron_raw_'+mouse_recday+'_0.npy'))

    #Importing Ephys
    print('Importing Ephys')
    num_sessions=len(np.load(Input_folder+'awake_session_behaviour_'+mouse_recday+'.npy'))
    ses_not_found=[]
    for session in np.arange(num_sessions):
        try:
            ephys_ = np.load(Input_folder+'Neuron_'+mouse_recday+'_'+str(session)+'.npy')
            exec('ephys_ses_'+str(session)+'_=ephys_')
        except:
            print('Ephys not found')
            ses_not_found.append(session)


    ##Calculating rotations
    print('Calculating rotations')
    rotation_dist_mat=np.zeros((num_neurons,num_phases,num_locations,num_sessions,num_sessions))
    rotation_angle_mat=np.zeros((num_neurons,num_phases,num_locations,num_sessions,num_sessions))
    mean_rotation_dist_all=np.zeros((num_neurons,num_phases,num_locations))
    num_passes_all=np.zeros((num_sessions,num_phases,num_locations))
    best_node_phase=np.zeros((num_neurons,2))

    mean_FRs_all=np.zeros((num_neurons,num_sessions,num_lags,num_locations_withedges))
    mean_FRs_shuff_all=np.zeros((num_neurons,num_sessions,num_iterations,num_locations_withedges))
    
    mean_FRs_all[:]=np.nan
    mean_FRs_shuff_all[:]=np.nan
    for neuron in np.arange(num_neurons):
        print(neuron)
        for session in np.arange(num_sessions):
            if session in ses_not_found:
                print('not found x')
                continue
            exec('ephys_=ephys_ses_'+str(session)+'_')
            location_mat_=np.load(Input_folder+'Location_'+mouse_recday+'_'+str(session)+'.npy')
            if len(location_mat_)==0:
                continue
            occupancy_mat=np.reshape(location_mat_,(4,len(location_mat_),len(location_mat_.T)//4))
            occupancy_conc=np.concatenate(location_mat_)
            ephys_neuron_=ephys_[neuron]
            neuron_mat=ephys_neuron_

            phase_mat=np.zeros(np.shape(occupancy_mat))
            phase_mat[:,:,30:60]=1
            phase_mat[:,:,60:90]=2
            phase_conc=np.concatenate(np.hstack(phase_mat))

            phase_peaks=np.load(Input_folder+'tuning_phase_boolean_max_'+mouse_recday+'.npy')[session]
            pref_phase_neurons=np.argmax(phase_peaks,axis=1)
            pref_phase=pref_phase_neurons[neuron]

            mean_FRs_neuronsession=np.zeros((num_lags,num_locations_withedges))
            mean_FRs_neuronsession[:]=np.nan
            mean_FRs_shuff_session=np.zeros((num_iterations,num_locations_withedges))
            mean_FRs_shuff_session[:]=np.nan

            if len(neuron_mat)==0 or len(occupancy_mat)==0:
                if neuron==0 and session==0:
                    print('No data for session'+str(session))

            else:

                neuron_conc=np.concatenate(neuron_mat)

                min_len=np.min([len(occupancy_conc),len(neuron_conc)])
                neuron_conc=neuron_conc[:min_len]
                occupancy_conc=occupancy_conc[:min_len]
                phase_conc=phase_conc[:min_len]

                ##Shifting
                for shift in np.arange(num_lags):
                    occupancy_shifted=np.roll(occupancy_conc,len_phase*shift) 
                    ###i.e. if peak in 4th bin (bin index 3) then anchor is 4 bins behind peak of activity 
                    occupancy_shifted_nonan=occupancy_shifted[~np.isnan(occupancy_shifted)]
                    neuron_conc_nonan=neuron_conc[~np.isnan(occupancy_shifted)]

                    occupancy_shifted_nonan_prefphase=occupancy_shifted[np.logical_and(~np.isnan(occupancy_shifted),\
                                                                           phase_conc==pref_phase)]
                    neuron_conc_nonan_pref_phase=neuron_conc[np.logical_and(~np.isnan(occupancy_shifted),\
                                                                            phase_conc==pref_phase)]

                    if use_prefphase==True:
                        mean_FRs=st.binned_statistic(occupancy_shifted_nonan_prefphase,\
                                                     neuron_conc_nonan_pref_phase,bins=np.arange(22)+1,statistic='mean')[0]
                    else:
                        mean_FRs=st.binned_statistic(occupancy_shifted_nonan, neuron_conc_nonan,bins=\
                                                 np.arange(num_locations_withedges+1)+1,statistic='mean')[0]

                    mean_FRs_neuronsession[shift]=mean_FRs

            mean_FRs_all[neuron][session]=mean_FRs_neuronsession

    Spatial_anchoring_dic['Phase_shifted_maps'][mouse_recday]=mean_FRs_all

In [ ]:
###Making Spatial lagged matrices (for plotting)
num_lags=12
num_locations=num_nodes=9
num_locations_withedges=21
num_lags=12
day_type='combined_ABCDonly'
for mouse_recday in np.load(Input_folder+day_type+'_days.npy'):
    non_repeat_ses=non_repeat_ses_maker(mouse_recday)
    num_neurons=len(np.load(Input_folder+'Neuron_raw_'+mouse_recday+'_0.npy'))
    
    node_rate_matrices=np.empty((num_neurons,len(non_repeat_ses),num_lags,3,3))
    node_rate_matrices[:]=np.nan

    edge_rate_matrices=np.empty((num_neurons,len(non_repeat_ses),num_lags,5,5))
    edge_rate_matrices[:]=np.nan

    node_edge_rate_matrices=np.empty((num_neurons,len(non_repeat_ses),num_lags,5,5))
    node_edge_rate_matrices[:]=np.nan
    for neuron in np.arange(num_neurons):
        maps_neuron=Spatial_anchoring_dic['Phase_shifted_maps'][mouse_recday][neuron]
        
        node_rate_mat=np.zeros((3,3))
        node_rate_mat[:]=np.nan
        
        edge_rate_mat=np.zeros((5,5))
        edge_rate_mat[:]=np.nan


        

        for awake_session_ind_ind, awake_session_ind in enumerate(non_repeat_ses):
            for lag in np.arange(num_lags):
                for node_indx,mat_indx in enumerate(Task_grid):
                    node_rate_matrices[neuron,awake_session_ind_ind,lag,mat_indx[0],mat_indx[1]]=\
                    node_rate_mat[mat_indx[0],mat_indx[1]]=\
                    maps_neuron[awake_session_ind,lag,node_indx]

                for edge_indx,mat_indx in enumerate(Edge_grid_coord2):
                    edge_rate_matrices[neuron,awake_session_ind_ind,lag,mat_indx[0],mat_indx[1]]=\
                    edge_rate_mat[mat_indx[0],mat_indx[1]]=\
                    maps_neuron[awake_session_ind,lag,num_nodes:][edge_indx]

                node_edge_mat=edge_node_fill(edge_rate_mat,node_rate_mat)

                node_edge_rate_matrices[neuron,awake_session_ind_ind,lag]=node_edge_mat
    
    Spatial_anchoring_dic['Phase_shifted_node_edge_matrices'][mouse_recday]=node_edge_rate_matrices
    Spatial_anchoring_dic['Phase_shifted_node_matrices'][mouse_recday]=node_rate_matrices

In [ ]:
np.load(Input_folder+'State_95'+mouse_recday+'.npy')
np.load(Input_folder+'State_95'+mouse_recday+'.npy')

In [ ]:
###Task-space Shifted spatial correlations
Phase_spatial_corr_dic=rec_dd()
run_shuffle=True
num_locations=9
num_locations_withedges=21
num_lags=12
len_phase=int(360/num_lags)
if run_shuffle==True:
    num_iterations=100
else:
    num_iterations=0
for mouse_recday in np.load(Input_folder+'combined_ABCDonly_days.npy'):
    num_sessions=len(np.load(Input_folder+'awake_session_behaviour_'+mouse_recday+'.npy'))
    num_neurons=len(np.load(Input_folder+'Neuron_raw_'+mouse_recday+'_0.npy'))
    
    print(mouse_recday)
    sessions=np.load(Input_folder+'Task_num_'+mouse_recday+'.npy')
    num_refses=len(np.unique(sessions))
    num_comparisons=num_refses-1
    repeat_ses=np.where(rank_repeat(sessions)>0)[0]
    non_repeat_ses=non_repeat_ses_maker(mouse_recday)  
    
    state_tuned=np.load(Input_folder+'State_95'+mouse_recday+'.npy')
    
    Phase_shifted_maps=Spatial_anchoring_dic['Phase_shifted_maps'][mouse_recday]
    
    len_lower_triangle=len(matrix_triangle(np.corrcoef(Phase_shifted_maps[0,non_repeat_ses,0]),direction='lower'))
    corrs_all=np.zeros((num_neurons,num_lags,len_lower_triangle)) 
    corrs_all_shuff=np.zeros((num_neurons,num_iterations,num_lags,len_lower_triangle))
    corrs_all_shuff_mean=np.zeros((num_neurons,num_iterations))
    corrs_crossval_all=np.zeros((num_neurons,num_lags,len(non_repeat_ses))) 
    
    corrs_all[:]=np.nan
    corrs_all_shuff[:]=np.nan
    corrs_all_shuff_mean[:]=np.nan
    corrs_crossval_all[:]=np.nan
    for neuron in np.arange(num_neurons):
        Phase_shifted_maps_clean=Phase_shifted_maps[neuron,non_repeat_ses,:,:num_locations]
        ###Note: removing edges from comparison as shifts cause systematic errors

        corrs_neuron=np.vstack(([matrix_triangle(pd.DataFrame.to_numpy((pd.DataFrame(Phase_shifted_maps_clean[:,ii].T))\
                        .corr()),direction='lower') for ii in range(num_lags)]))
        
        corrs_all[neuron]=corrs_neuron
        
        corrs_neuron_crossval=np.vstack(([[pd.DataFrame(np.nanmean(Phase_shifted_maps_clean[np.setdiff1d\
                                (np.arange(len(non_repeat_ses)),session),lag],axis=0))[0]\
                         .corr(pd.DataFrame(Phase_shifted_maps_clean[session,lag])[0]) for lag in \
                        np.arange(np.shape(Phase_shifted_maps_clean)[1])] for session in np.arange(len(non_repeat_ses))]))
        
        corrs_crossval_all[neuron]=corrs_neuron_crossval.T
        
        ###logic of shuffle, randomly shuffling spatial maps across lags within a task, so that across task comparison
        ##is between maps at different lags (e.g. lag 6 in task 1 against lag 3 in task 2 and lag 0 in task 3...etc)
        ##then take mean correlation per iteration, then take 95th percentile of the means across iterations
        ##so one threshold per neuron
        for iteration in np.arange(num_iterations):
            Phase_shifted_maps_clean_copy=np.copy(Phase_shifted_maps_clean)
            [np.random.shuffle(Phase_shifted_maps_clean_copy[ii]) for ii in range(len(Phase_shifted_maps_clean_copy))]
            corrs_neuron_shuff=np.vstack(([matrix_triangle(pd.DataFrame.to_numpy((\
            pd.DataFrame(Phase_shifted_maps_clean_copy[:,ii].T)).corr()),direction='lower') for ii in range(num_lags)]))
        
            corrs_all_shuff[neuron][iteration]=corrs_neuron_shuff
            corrs_all_shuff_mean[neuron][iteration]=np.nanmean(corrs_neuron_shuff)
            
    
    mean_corrs=np.nanmean(corrs_all,axis=2)
    std_corrs=np.nanstd(corrs_all,axis=2)
    max_corr_bin=_nanargmax(mean_corrs,axis=1)
    
    mean_corrs_crossval=np.nanmean(corrs_crossval_all,axis=2)
    max_corr_bin_crossval=_nanargmax(mean_corrs_crossval,axis=1)
    
    Phase_spatial_corr_dic['corrs_all'][mouse_recday]=corrs_all
    Phase_spatial_corr_dic['Max_corr_bin'][mouse_recday]=max_corr_bin
    
    if run_shuffle==True:
        
        thr_corrs=np.percentile(corrs_all_shuff_mean,95,axis=1)
        Phase_spatial_corr_dic['Threshold'][mouse_recday]=thr_corrs
        Phase_spatial_corr_dic['corrs_all_shuff'][mouse_recday]=corrs_all_shuff

    Phase_spatial_corr_dic['corrs_crossval_all'][mouse_recday]=corrs_crossval_all
    Phase_spatial_corr_dic['Max_corr_bin_crossval'][mouse_recday]=max_corr_bin_crossval


In [ ]:
###Task-space Shifted spatial correlations - cross-validated single correlation value per neuron

close_to_anchor_bins30=[0,11]
close_to_anchor_bins90=[0,1,2,11,10,9]

include_bridges=False

for mouse_recday in np.load(Input_folder+'combined_ABCDonly_days.npy'):
    num_sessions=len(np.load(Input_folder+'awake_session_behaviour_'+mouse_recday+'.npy'))
    num_neurons=len(np.load(Input_folder+'Neuron_raw_'+mouse_recday+'_0.npy'))
    
    print(mouse_recday)
    
    sessions=np.load(Input_folder+'Task_num_'+mouse_recday+'.npy')
    num_refses=len(np.unique(sessions))
    num_comparisons=num_refses-1
    repeat_ses=np.where(rank_repeat(sessions)>0)[0]
    non_repeat_ses=non_repeat_ses_maker(mouse_recday)  
    
    Phase_boolean=np.load(Input_folder+'Phase_'+mouse_recday+'.npy')
    State_boolean=np.load(Input_folder+'State_95'+mouse_recday+'.npy')
    
    used_boolean=np.logical_and(Phase_boolean,State_boolean)

    
    Phase_shifted_maps_=Spatial_anchoring_dic['Phase_shifted_maps'][mouse_recday]
    mean_bestlag_all=np.zeros((num_neurons)) 
    mean_bestlag_nonspatial=np.zeros((num_neurons))
    mean_bestlag_nonspatial_strict=np.zeros((num_neurons))
    max_lags_all=np.zeros((num_neurons,len(non_repeat_ses)))
    mean_bestlag_all[:]=np.nan
    mean_bestlag_nonspatial[:]=np.nan
    mean_bestlag_nonspatial_strict[:]=np.nan
    max_lags_all[:]=np.nan

    
    for neuron in np.arange(num_neurons):
        if include_bridges==True:
            Phase_shifted_map=Phase_shifted_maps_[neuron,non_repeat_ses]
        
        else:
            Phase_shifted_map=Phase_shifted_maps_[neuron,non_repeat_ses,:,:num_locations]
        ###Note: removing edges from comparison as shifts cause systematic errors


        max_lags=[_nanargmax([np.nanmean(matrix_triangle(\
            pd.DataFrame.to_numpy(pd.DataFrame(Phase_shifted_map[\
                np.setdiff1d(np.arange(len(non_repeat_ses)),session),lag]).T.corr()))) for\
                    lag in np.arange(np.shape(Phase_shifted_map)[1])])\
                        for session in np.arange(len(non_repeat_ses))]

        max_corrs=[np.nanmax([np.nanmean(matrix_triangle(\
            pd.DataFrame.to_numpy(pd.DataFrame(Phase_shifted_map[\
                np.setdiff1d(np.arange(len(non_repeat_ses)),session),lag]).T.corr()))) for\
                    lag in np.arange(np.shape(Phase_shifted_map)[1])])\
                        for session in np.arange(len(non_repeat_ses))]

        mean_bestlag=np.nanmean([np.nanmean(pd.DataFrame(Phase_shifted_map[:,max_lags[session]]).T.corr()\
             [session][np.setdiff1d(np.arange(len(non_repeat_ses)),session)])\
                                 if np.isnan(max_lags[session])==False else np.nan
                                 for session in np.arange(len(non_repeat_ses))])
        
        
        
        mean_best_nonspatial=np.nanmean([np.nanmean(pd.DataFrame(Phase_shifted_map[:,max_lags[session]]).T.corr()\
                 [session][np.setdiff1d(np.arange(len(non_repeat_ses)),session)])\
                     if max_lags[session] not in close_to_anchor_bins30 and np.isnan(max_lags[session])==False\
                                         else np.nan for session in np.arange(len(non_repeat_ses))])
        
        mean_best_nonspatial_strict=np.nanmean([np.nanmean(pd.DataFrame(Phase_shifted_map[:,max_lags[session]]).T.corr()\
                 [session][np.setdiff1d(np.arange(len(non_repeat_ses)),session)])\
                     if max_lags[session] not in close_to_anchor_bins90 and np.isnan(max_lags[session])==False 
                                                else np.nan for session in np.arange(len(non_repeat_ses))])

        
        ##removing spatial neurons
        ##overwrites above as otherwise contaminating nonspatial with imperfections of spatial neurons
        if st.mode(max_lags,keepdims=True)[0][0] in close_to_anchor_bins30 and st.mode(max_lags,keepdims=True)[1][0]>1:
            mean_best_nonspatial=np.nan
            
        if st.mode(max_lags,keepdims=True)[0][0] in close_to_anchor_bins90 and st.mode(max_lags,keepdims=True)[1][0]>1:
            mean_best_nonspatial_strict=np.nan
            
        
        mean_bestlag_all[neuron]=mean_bestlag
        mean_bestlag_nonspatial[neuron]=mean_best_nonspatial
        mean_bestlag_nonspatial_strict[neuron]=mean_best_nonspatial_strict
        
        max_lags_all[neuron]=max_lags
    
    Phase_spatial_corr_dic['mean_bestlag_corr_crossval'][mouse_recday]=mean_bestlag_all
    Phase_spatial_corr_dic['mean_bestlag_corr_nonzerolag_crossval'][mouse_recday]=mean_bestlag_nonspatial
    Phase_spatial_corr_dic['mean_bestlag_corr_nonzerolag_strict_crossval'][mouse_recday]=mean_bestlag_nonspatial_strict
    Phase_spatial_corr_dic['max_lags_all'][mouse_recday]=max_lags_all


In [ ]:
use_tuned=True

used_recdays_=np.asarray(list(Phase_spatial_corr_dic['mean_bestlag_corr_crossval'].keys()))



state_tuning=np.hstack(([np.load(Input_folder+'State_'+mouse_recday+'.npy')\
                                 for mouse_recday in used_recdays_]))


place_tuning=np.hstack(([np.load(Input_folder+'Place_'+mouse_recday+'.npy')\
        for mouse_recday in used_recdays_]))

neurons_tuned=state_tuning

plt.rcParams["figure.figsize"] = (7,5)
plt.rcParams['axes.linewidth'] = 4
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.left'] = True
plt.rcParams['axes.spines.bottom'] = True
for subset in ['mean_bestlag_corr_crossval','mean_bestlag_corr_nonzerolag_crossval',\
               'mean_bestlag_corr_nonzerolag_strict_crossval']:
    print(subset)
    if use_tuned==True:
        mean_bestlag_corr_crossval=\
        remove_nan(np.hstack(([Phase_spatial_corr_dic[subset][mouse_recday] for mouse_recday in used_recdays_]))\
                   [neurons_tuned])
    else:
        mean_bestlag_corr_crossval=\
        remove_nan(np.hstack(([Phase_spatial_corr_dic[subset][mouse_recday] for mouse_recday in used_recdays_])))
        
    plt.hist(mean_bestlag_corr_crossval,bins=50,color='grey')
    plt.axvline(0,color='black',ls='dashed')
    plt.tick_params(axis='both',  labelsize=20)
    plt.tick_params(width=2, length=6)
    plt.savefig(Output_folder+'SpatialLag_analysis_'+subset+'.svg' , bbox_inches = 'tight', pad_inches = 0)
    plt.show()
    print(len(mean_bestlag_corr_crossval))
    print(st.ttest_1samp(mean_bestlag_corr_crossval,0))
    print('')

In [ ]:
###Anchoring in left out session - single anchor 

'''
The logic here is as follows:
-train-test splits
-for each possible test session find the best anchor by searching each possible reference session and finding the
reference session that gives the anchor with the lowest mean distance across all training sessions (i.e. sessions that
arent reference or test sessions)
-this gives a single anchor per training-test split per neuron which is calculated exclusively from training sessions - 
hence no double-dipping

to see that this isnt biased - set shuffle_angles to True - can see uniform distribution of angles
'''


thr_tuning_sessions=0
coh_thr=1-math.cos(np.deg2rad(45))
angles_spatialanchor_num_allrefs=[]

thr_anchored=0.5

day_type='combined_ABCDonly'


phase_random=False
shuffle_angles=False

delete_old=False


for mouse_recday in np.load(Input_folder+day_type+'_days.npy'):
    print(mouse_recday)
    
    if day_type=='combined_ABCDonly':
        num_states=4

    
    coh_thr1=(360/num_states)//2
    coh_thr2=360-coh_thr1
    
    num_bins=num_states*90
    angle_correction=360/num_bins ### converts bin value into angles (used when number of states is not 4)
    num_trials=np.load(Input_folder+'Num_trials_'+mouse_recday+'.npy',\
                                        allow_pickle=True)
    trials_completed_ses=np.where(num_trials>2)[0]
    
    num_sessions=len(np.load(Input_folder+'awake_session_behaviour_'+mouse_recday+'.npy'))
    num_neurons=len(np.load(Input_folder+'Neuron_raw_'+mouse_recday+'_0.npy')) 
    
    print(num_sessions)
    sessions=np.load(Input_folder+'Task_num_'+mouse_recday+'.npy')
        
    repeat_ses=np.where(rank_repeat(sessions)>0)[0]
    non_repeat_ses=non_repeat_ses_maker(mouse_recday)  ###this defines the sessions used
    ###only the first session from each task is used
    

    
    non_repeat_ses=np.intersect1d(non_repeat_ses,trials_completed_ses)
    
    num_comparisons=len(non_repeat_ses)-1
    num_refses=len(non_repeat_ses)
    
    angles_=np.copy(Spatial_anchoring_dic['rotation_angle_mat'][mouse_recday])*angle_correction
    
    dists_all=np.zeros((num_refses,num_refses,num_neurons)) 
    angles_all=np.zeros((num_refses,num_refses,num_neurons)) 
    Best_anchor_all=np.zeros((num_refses,num_refses,num_neurons,2)) 
    Mins_all=np.zeros((num_refses,num_refses,num_neurons)) 
    
    dists_all[:]=np.nan
    angles_all[:]=np.nan
    Best_anchor_all[:]=np.nan
    Mins_all[:]=np.nan

    if shuffle_angles == True:        
        angles_shuff=np.copy(angles_)*angle_correction
        angles_shuff=(angles_shuff+numpy.random.random_integers(1, num_states, np.shape(angles_shuff))*90)%num_bins
        ###i.e. adding random number thats a multiple of 90 degrees - to keep phase tuning 
        anglesX=angles_shuff
        
        
        
        dists_shuff=1-np.cos(np.deg2rad(angles_shuff))
        distsX=dists_shuff
    else:
        anglesX=angles_
        distsX=1-np.cos(np.deg2rad(angles_))

    for ses_reference_ind, ses_reference in enumerate(non_repeat_ses):   
        non_repeat_ses_noref=np.sort(np.setdiff1d(non_repeat_ses,ses_reference)) 
        ###sessions used that arent the reference session

        ###cross-validating against lagged spatial map analysis 
        dists_day=distsX[:,:,:,ses_reference,non_repeat_ses_noref] 
        angles_day=anglesX[:,:,:,ses_reference,non_repeat_ses_noref] 
        num_neurons=len(np.load(Input_folder+'Neuron_raw_'+mouse_recday+'_0.npy')) 
        
        ###i.e. for each test-train split will get one best anchor 
        for Test_comp in np.arange(num_comparisons): 
            Test_ind=np.setdiff1d(np.arange(num_refses),ses_reference_ind)[Test_comp] ##used for indexing arrays
            
            Training_comp=np.sort(np.setdiff1d(np.arange(num_comparisons),Test_comp)) 
            Training=dists_day[:,:,:,Training_comp] 
            Test=dists_day[:,:,:,Test_comp] 
            Test_angles=angles_day[:,:,:,Test_comp] 

            mean_Training=np.nanmean(Training,axis=3)
            ##mean distance for the training sessions in this test-train split
            
            max_Training=np.nanmax(Training,axis=3)
            ##max distance for the training sessions in this test-train split
            
            XXX_training=mean_Training

            Best_anchor_=np.asarray([[np.where(x == np.nanmin(x))[0][0],np.where(x == np.nanmin(x))[1][0]] 
                        if ~np.isnan(np.nanmean(x)) else [np.nan,np.nan] for neuron,x in enumerate(XXX_training)])
            
            Best_anchor_randphase=np.copy(Best_anchor_)
            Best_anchor_randphase[:,0]=np.asarray(Best_anchor_[:,0]+random.choices(np.arange(2)+1,k=len(Best_anchor_)))%3
            
            if phase_random==True:
                Best_anchor=Best_anchor_randphase
            else:
                Best_anchor=Best_anchor_
                
            
            Mins=np.asarray([np.nanmin(x) for neuron,x in enumerate(XXX_training)])
            ###smallest mean distance (i.e. distance belonging to best anchor) 
            ##for the training sessions in this test-train split 

            
            ##this is where the distances/angles for the test session relative to the reference session is calculated
            dists=np.asarray([Test[neuron][int(Best_anchor[neuron][0]),int(Best_anchor[neuron][1])] 
                              if ~np.isnan(np.nanmean(x)) else np.nan for neuron,x in enumerate(XXX_training)]) 
            angles=np.asarray([Test_angles[neuron][int(Best_anchor[neuron][0]),int(Best_anchor[neuron][1])] 
                               if ~np.isnan(np.nanmean(x)) else np.nan for neuron,x in enumerate(XXX_training)]) 

            
            dists_all[ses_reference_ind,Test_ind]=dists 
            angles_all[ses_reference_ind,Test_ind]=angles 
            Best_anchor_all[ses_reference_ind,Test_ind]=Best_anchor 
            Mins_all[ses_reference_ind,Test_ind]=Mins
            ## for each neuron, a row is a reference session and a column is a training-test split (indexed
            ##by the test session), note that the number reported is the mean distance for best anchor over only 
            ##the training sessions (see Mins)
            ##so e.g. row 0 column 1 is the mean distance over the training sessions training-test split where test is 
            #session 1 (second session) and reference session is session 0 (first session)


    if np.isnan(np.nanmean(dists_all))==True:
        continue
    min_ses=np.nanargmin(Mins_all,axis=0)
    ### this is the reference session that has the best training distances for each test-train split and each neuron
    
    dists_test=np.asarray([[dists_all[min_ses[ses,neuron],ses,neuron]\
                        for ses in range(len(dists_all[:,:,neuron].T))]\
                       for neuron in range(num_neurons)]).T

    angles_test=np.asarray([[angles_all[min_ses[ses,neuron],ses,neuron]\
                            for ses in range(len(dists_all[:,:,neuron].T))]\
                           for neuron in range(num_neurons)]).T
    
    Best_anchor_test=np.asarray([[Best_anchor_all[min_ses[ses,neuron],ses,neuron]\
                            for ses in range(len(dists_all[:,:,neuron].T))]\
                           for neuron in range(num_neurons)]).T
    
    ##tuned sessions
    
    sum_statepeaks=np.sum(np.load(Input_folder+'tuning_state_boolean_'+mouse_recday+'.npy'),axis=2).T
    sum_statepeaks=sum_statepeaks[:,non_repeat_ses]
    state_bool_=sum_statepeaks>0
    state_bool_
    
    ##finding most common  anchor for each neuron - Note dont use this when needing cross-validation   
    most_common_anchor=\
    np.vstack(([most_common_pair(Best_anchor_test[:,:,neuron].T)[0] for neuron in range(num_neurons)]))

    most_common_anchor_bool=np.vstack(([[np.all(Best_anchor_test[:,ses,neuron]==most_common_anchor[neuron])\
                 for ses in range(len(min_ses[:,neuron]))] for neuron in np.arange(num_neurons)]))

    freq_most_common_anchor=\
    np.hstack(([most_common_pair(Best_anchor_test[:,:,ii].T)[1] for ii in range(num_neurons)]))
    Anchored_bool_half=freq_most_common_anchor>=thr_anchored
    Anchored_bool_always=freq_most_common_anchor==1

    Min_dist=np.vstack(([[Mins_all[min_ses[ses,neuron],ses,neuron] for ses in range(len(min_ses[:,neuron]))]\
                         for neuron in np.arange(num_neurons)]))
    max_mindist=np.nanmax(Min_dist,axis=1)

    max_mindist_most_common_anchor=np.asarray([np.nanmax(Min_dist[neuron][most_common_anchor_bool[neuron]]) if\
     len(Min_dist[neuron][most_common_anchor_bool[neuron]])>0 else np.nan for neuron in range(num_neurons)])

    Anchored_bool_coherent_most_common_anchor=max_mindist_most_common_anchor<coh_thr

    Anchored_bool_coherent=max_mindist<coh_thr

    Anchored_bool=np.logical_and(Anchored_bool_half,Anchored_bool_coherent_most_common_anchor)
    Anchored_bool_strict=np.logical_and(Anchored_bool_always,Anchored_bool_coherent)
    
    
    #####
    most_common_anchor_tuned=np.vstack(([most_common_pair(Best_anchor_test[:,state_bool_[neuron],neuron].T)[0]\
                          if len(Best_anchor_test[:,state_bool_[neuron],neuron].T)>1 else [np.nan,np.nan]\
                          for neuron in range(num_neurons)]))

    most_common_anchor_bool_tuned_allses=np.vstack(([[np.all(Best_anchor_test[:,ses,neuron]\
                                                      ==most_common_anchor_tuned[neuron])\
                 for ses in range(len(min_ses[:,neuron]))] if np.isnan(most_common_anchor_tuned[neuron,0])==False\
                                              else np.repeat(False, len(min_ses[:,neuron]))
                                              for neuron in np.arange(num_neurons)]))

    most_common_anchor_bool_tuned=[[np.all(Best_anchor_test[:,ses,neuron]==most_common_anchor_tuned[neuron])\
                 for ses in np.where(state_bool_[neuron]==True)[0]] if np.isnan(most_common_anchor_tuned[neuron,0])==False\
                                              else [] for neuron in np.arange(num_neurons)]

    most_common_anchor_bool_tuned_ses=[np.where(state_bool_[neuron]==True)[0] if \
                                       np.isnan(most_common_anchor_tuned[neuron,0])==False else [] \
                                       for neuron in np.arange(num_neurons)]

    most_common_anchor_bool_tuned_ses_common=[np.where(np.logical_and(state_bool_[neuron]==True,\
                                                                most_common_anchor_bool_tuned_allses[neuron]==True))[0] if \
                                       np.isnan(most_common_anchor_tuned[neuron,0])==False else [] \
                                       for neuron in np.arange(num_neurons)]

    Min_dist_tuned=[[Mins_all[min_ses[ses,neuron],ses,neuron] for ses in \
                     most_common_anchor_bool_tuned_ses_common[neuron]] if \
                    len(most_common_anchor_bool_tuned_ses_common[neuron])>1 else [] for neuron in np.arange(num_neurons)]

    max_mindist_tuned=np.asarray([np.max(Min_dist_tuned[neuron]) if len(Min_dist_tuned[neuron])>1 else np.nan\
                      for neuron in np.arange(num_neurons)])
    
    proportion_mostcommon_tuned=np.asarray([np.sum(most_common_anchor_bool_tuned[neuron])\
                                      /len(most_common_anchor_bool_tuned[neuron])
    if len(most_common_anchor_bool_tuned[neuron])>1 else np.nan for neuron in range(num_neurons)])

    Anchored_bool_tuned_half=proportion_mostcommon_tuned>thr_anchored
    Anchored_bool_tuned_always=proportion_mostcommon_tuned==1
    Anchored_bool_coherent_tuned=max_mindist_tuned<coh_thr
    
    Anchored_bool_tuned=np.logical_and(Anchored_bool_tuned_half,Anchored_bool_coherent_tuned)
    Anchored_bool_tuned_strict=np.logical_and(Anchored_bool_tuned_always,Anchored_bool_coherent_tuned)
    
    ###cross-validated anchored booleans   
    most_common_anchor_crossval=\
    np.vstack(([np.vstack(([most_common_pair\
                            (Best_anchor_test[:,[ses_ind!=ind for ses_ind in range(len(min_ses))],ii].T)[0]\
                            for ii in range(num_neurons)])) for ind in range(len(min_ses))]))

    freq_most_common_anchor_crossval=\
    np.vstack(([np.hstack(([most_common_pair\
                            (Best_anchor_test[:,[ses_ind!=ind for ses_ind in range(len(min_ses))],ii].T)[1]\
                            for ii in range(num_neurons)])) for ind in range(len(min_ses))]))

    max_mindist_cross_val=np.vstack(([np.max(Min_dist[:,[ses_ind!=ind for ses_ind in range(len(min_ses))]],axis=1)\
            for ind in range(len(min_ses))]))
    Anchored_bool_coherent_crossval=max_mindist_cross_val<coh_thr    

    Anchored_bool_half_crossval=freq_most_common_anchor_crossval>=thr_anchored
    Anchored_bool_always_crossval=freq_most_common_anchor_crossval==1

    Anchored_bool_crossval=np.logical_and(Anchored_bool_half_crossval,Anchored_bool_coherent_crossval)
    Anchored_bool_strict_crossval=np.logical_and(Anchored_bool_always_crossval,Anchored_bool_coherent_crossval)

    ##Neuron by neuron coherence 
    coh_bool=dists_test<coh_thr 
    coh_bool_sum=np.sum(coh_bool,axis=0) 
 
    neurons_tuned=np.where(np.load(Input_folder+'State_95'+mouse_recday+'.npy')==True)[0] ##state neurons

        
    
    
    if len(neurons_tuned)==0:
        if delete_old==True:
            del(Spatial_anchoring_dic['MeanAngles_spatial_Anchor_best'][mouse_recday])
            del(Spatial_anchoring_dic['best_node_phase_used'][mouse_recday])
            del(Spatial_anchoring_dic['most_common_anchor'][mouse_recday])
            del(Spatial_anchoring_dic['most_common_anchor_crossval'][mouse_recday])
            del(Spatial_anchoring_dic['Anchored_bool'][mouse_recday])
            del(Spatial_anchoring_dic['Anchored_bool_strict'][mouse_recday])
            del(Spatial_anchoring_dic['Anchored_bool_crossval'][mouse_recday])
            del(Spatial_anchoring_dic['Anchored_bool_strict_crossval'][mouse_recday])   
            del(Spatial_anchoring_dic['Angles_all'][mouse_recday])
            del(Spatial_anchoring_dic['Dists_all'][mouse_recday])
            del(Spatial_anchoring_dic['Coherent_proportion'][mouse_recday])
            del(Spatial_anchoring_dic['Coherent_proportion'][day_type][mouse_recday])
            del(Spatial_anchoring_dic['Neuron_coherence'][mouse_recday])
            del(Spatial_anchoring_dic['Best_reference_task'][mouse_recday])
            del(Spatial_anchoring_dic['Neuron_tuned'][mouse_recday])
            del(Spatial_anchoring_dic['Neuron_used_histogram'][mouse_recday])
        print('Not used - no tuned neurons')
        continue

    neurons_used=neurons_tuned ##ignoring number of peaks and just saying is the neuron tuned in atleast one task
   
        

    dists_used=dists_test[:,neurons_used] 
    angles_used=angles_test[:,neurons_used]
    
    ###making histograms by averaging across all training-test splits
    angles_spatialanchor_num_all=[] 
    angles_spatialanchor_cohprop_all=[] 
    for ii in np.arange(num_comparisons): 
        angles_spatialanchor=remove_nan(angles_used[ii]) 
        if len(angles_spatialanchor)>0: 
            coh_prop=len(np.where(np.logical_or(angles_spatialanchor<coh_thr1 ,angles_spatialanchor>coh_thr2))[0])\
            /len(angles_spatialanchor) 
            angles_spatialanchor_num=np.histogram(angles_spatialanchor,np.linspace(0,360,37))[0] 
        else: 
            coh_prop=np.nan 
            angles_spatialanchor_num=np.repeat(np.nan,36) 

        angles_spatialanchor_cohprop_all.append(coh_prop) 
        angles_spatialanchor_num_all.append(angles_spatialanchor_num) 
    angles_spatialanchor_num_mean=np.nanmean(np.asarray(angles_spatialanchor_num_all),axis=0) 
    angles_spatialanchor_cohprop_mean=np.nanmean(angles_spatialanchor_cohprop_all) 

    polar_plot_stateX(angles_spatialanchor_num_mean,angles_spatialanchor_num_mean, 
                      angles_spatialanchor_num_mean,color='black',labels='angles',plot_type='bar') 
    plt.show() 
    
    
    
    Spatial_anchoring_dic['MeanAngles_spatial_Anchor_best'][mouse_recday]=angles_spatialanchor_num_mean 
    Spatial_anchoring_dic['best_node_phase_used'][mouse_recday]=Best_anchor_test
    Spatial_anchoring_dic['most_common_anchor'][mouse_recday]=most_common_anchor
    Spatial_anchoring_dic['most_common_anchor_bool'][mouse_recday]=most_common_anchor_bool
    Spatial_anchoring_dic['most_common_anchor_crossval'][mouse_recday]=most_common_anchor_crossval
    Spatial_anchoring_dic['Anchored_bool'][mouse_recday]=Anchored_bool
    Spatial_anchoring_dic['Anchored_bool_strict'][mouse_recday]=Anchored_bool_strict
    Spatial_anchoring_dic['Anchored_bool_crossval'][mouse_recday]=Anchored_bool_crossval
    Spatial_anchoring_dic['Anchored_bool_strict_crossval'][mouse_recday]=Anchored_bool_strict_crossval 
    Spatial_anchoring_dic['Anchored_bool_tuned'][mouse_recday]=Anchored_bool_tuned
    Spatial_anchoring_dic['Anchored_bool_tuned_strict'][mouse_recday]=Anchored_bool_tuned_strict   
    Spatial_anchoring_dic['Angles_all'][mouse_recday]=angles_test 
    Spatial_anchoring_dic['Dists_all'][mouse_recday]=dists_test 
    Spatial_anchoring_dic['Coherent_proportion'][mouse_recday]=angles_spatialanchor_cohprop_mean 
    Spatial_anchoring_dic['Neuron_coherence'][mouse_recday]=(coh_bool).astype(int) 
    Spatial_anchoring_dic['Best_reference_task'][mouse_recday]=min_ses
    Spatial_anchoring_dic['Neuron_tuned'][mouse_recday]=neurons_tuned
    Spatial_anchoring_dic['Neuron_used_histogram'][mouse_recday]=neurons_used 



In [ ]:
print('') 
print('__________________') 
print('All recording days') 
#angles_spatialanchor_num=np.nansum(remove_empty(dict_to_array(Spatial_anchoring_dic['MeanAngles_spatial_Anchor_best']))\
#                                   ,axis=0) 


angles_spatialanchor_num=np.nansum(np.vstack(([Spatial_anchoring_dic['MeanAngles_spatial_Anchor_best'][mouse_recday]\
            for mouse_recday in np.load(Input_folder+day_type+'_days.npy')
                                    if len(Spatial_anchoring_dic['MeanAngles_spatial_Anchor_best'][mouse_recday])>0
                                               ])),axis=0)
print(angles_spatialanchor_num) 

polar_plot_stateX(angles_spatialanchor_num,angles_spatialanchor_num, 
                  angles_spatialanchor_num,color='black',labels='angles',plot_type='bar') 
plt.tick_params(axis='both',  labelsize=20)
plt.tick_params(width=2, length=6)
plt.savefig(Output_folder+'/Spatial_anchoring_histogram.svg', bbox_inches = 'tight', pad_inches = 0)
plt.show()

In [ ]:
total_coh=[]
total_neurons=[]
for mouse_recday in np.load(Input_folder+day_type+'_days.npy'):
    print(mouse_recday)
    if len(Spatial_anchoring_dic['MeanAngles_spatial_Anchor_best'][mouse_recday])==0:
        print('Not used')
        continue
        
    total_coh.append(Spatial_anchoring_dic['Coherent_proportion'][mouse_recday]*\
    len(Spatial_anchoring_dic['Neuron_used_histogram'][mouse_recday]))
    total_neurons.append(len(Spatial_anchoring_dic['Neuron_used_histogram'][mouse_recday]))
    
print('Total coherent proportion: '+str(np.sum(total_coh)/np.sum(total_neurons)))

print(two_proportions_test(np.sum(total_coh), np.sum(total_neurons),\
                           np.sum(total_neurons)*(1/num_states), np.sum(total_neurons)))
print(np.sum(total_coh), np.sum(total_neurons))

In [ ]:
###Simple anchor analysis 
apply_minimum_sesnumber=True
num_ses_thr=3 ##minimum number of comparisons for anchor to be considered
num_phases=3
num_nodes=9
for mouse_recday in np.load(Input_folder+day_type+'_days.npy'):
    print(mouse_recday)

    num_sessions=len(np.load(Input_folder+'awake_session_behaviour_'+mouse_recday+'.npy'))
    num_neurons=len(np.load(Input_folder+'Neuron_raw_'+mouse_recday+'_0.npy')) 
    non_repeat_ses=non_repeat_ses_maker(mouse_recday)
    dists_=np.copy(Spatial_anchoring_dic['rotation_dist_mat'][mouse_recday])
    num_trials=np.load(Input_folder+'Num_trials_'+mouse_recday+'.npy')
    
    trials_completed_ses=np.where(num_trials>2)[0]
    non_repeat_ses=np.intersect1d(non_repeat_ses,trials_completed_ses)
    
    best_anchor=np.zeros((num_neurons,2))
    best_anchor_refses=np.zeros(num_neurons)
    best_anchor_mindist=np.zeros(num_neurons)
    
    best_anchor[:]=np.nan
    best_anchor_refses[:]=np.nan
    best_anchor_mindist[:]=np.nan
    for neuron in np.arange(num_neurons):
        dists_neuron=dists_[neuron]
        dists_neuron_nonrepeat=dists_neuron[:,:,non_repeat_ses][:,:,:,non_repeat_ses]
        dists_neuron_nonrepeat_copy=np.copy(dists_neuron_nonrepeat)
        
        if apply_minimum_sesnumber==False:
            min_ref_ses=np.vstack(([[np.nanargmin(np.nanmean(fill_diagonal(dists_neuron_nonrepeat_copy\
                                                                           [phase_,location_],np.nan),axis=0))\
                                      for phase_ in np.arange(num_phases)] for location_ in np.arange(num_nodes)])).T

            min_value=np.vstack(([[np.nanmin(np.nanmean(fill_diagonal(dists_neuron_nonrepeat_copy\
                                                                      [phase_,location_],np.nan),axis=0))\
                                      for phase_ in np.arange(num_phases)] for location_ in np.arange(num_nodes)])).T
        else:
            min_ref_ses=np.zeros((num_phases,num_locations))
            min_value=np.zeros((num_phases,num_locations))
            min_ref_ses[:]=np.nan
            min_value[:]=np.nan

            for phase_ in np.arange(num_phases):
                for location_ in np.arange(num_nodes):
                    refses_passing_thr=np.count_nonzero(~np.isnan\
                                                        (fill_diagonal(dists_neuron_nonrepeat_copy[phase_,location_]\
                                                                       ,np.nan)),axis=0)>=num_ses_thr
                    means=np.nanmean(fill_diagonal(dists_neuron_nonrepeat_copy[phase_,location_],np.nan),axis=0)
                    means[refses_passing_thr==False]=np.nan
                    if np.isnan(np.nanmean(means))==False:
                        min_ref_ses_anchor=np.nanargmin(means)
                        min_value_anchor=np.nanmin(means)
                    else:
                        min_ref_ses_anchor=np.nan
                        min_value_anchor=np.nan

                    min_ref_ses[phase_,location_]=min_ref_ses_anchor
                    min_value[phase_,location_]=min_value_anchor

        overall_min_dist_value=np.nanmin(min_value)
        pref_phase=np.where(min_value==overall_min_dist_value)[0][0]
        pref_location=np.where(min_value==overall_min_dist_value)[1][0]
        pref_ref_ses=min_ref_ses[pref_phase,pref_location]
        
        best_anchor[neuron]=pref_phase,pref_location
        best_anchor_refses[neuron]=pref_ref_ses
        best_anchor_mindist[neuron]=overall_min_dist_value
        
    Spatial_anchoring_dic['Best_anchor_all'][mouse_recday]=best_anchor
    Spatial_anchoring_dic['Best_anchor_all_refses'][mouse_recday]=best_anchor_refses
    Spatial_anchoring_dic['Best_anchor_all_mindist'][mouse_recday]=best_anchor_mindist

In [ ]:
###Cross-validated single anchor correlation at best anchor
thr_visit=2
num_bins_state=90
num_states=4
for mouse_recday in np.load(Input_folder+day_type+'_days.npy'):

    print(mouse_recday)
    

    
    num_bins=num_states*90
    
    num_sessions=len(np.load(Input_folder+'awake_session_behaviour_'+mouse_recday+'.npy'))
    num_neurons=len(np.load(Input_folder+'Neuron_raw_'+mouse_recday+'_0.npy'))
    sessions=np.load(Input_folder+'Task_num_'+mouse_recday+'.npy')
    num_refses=len(np.unique(sessions))
    num_comparisons=num_refses-1
    repeat_ses=np.where(rank_repeat(sessions)>0)[0]
    non_repeat_ses=non_repeat_ses_maker(mouse_recday) 
    
    num_trials=np.load(Input_folder+'Num_trials_'+mouse_recday+'.npy')
    
    trials_completed_ses=np.where(num_trials>2)[0]
    non_repeat_ses=np.intersect1d(non_repeat_ses,trials_completed_ses)
    
  
    state_corrs_allneurons=np.zeros(num_neurons)
    state_corrs_allneurons[:]=np.nan
    
    for session in np.arange(num_sessions):
        try:

            ephys_ = np.load(Input_folder+'Neuron_'+mouse_recday+'_'+str(session)+'.npy')
            exec('ephys_ses_'+str(session)+'_=ephys_')
        except:
            print('Files not found')
            continue

    
    
    phase_peaks=np.load(Input_folder+'tuning_phase_boolean_max_'+mouse_recday+'.npy')[0]
        

    pref_phase_neurons=np.argmax(phase_peaks,axis=1)
    
    for neuron in np.arange(num_neurons):
        pref_phase=pref_phase_neurons[neuron]
        mean_smooth_all=np.zeros((len(non_repeat_ses),2,num_bins))
        mean_smooth_all[:]=np.nan
        for ses_ind, session in enumerate(non_repeat_ses):  
            try:
                
                best_anchor_phase_node=Spatial_anchoring_dic['best_node_phase_used'][mouse_recday][:,:,neuron][:,ses_ind]
                ref_task=Spatial_anchoring_dic['Best_reference_task'][mouse_recday][ses_ind,neuron]

                phase_=int(best_anchor_phase_node[0])
                location=int(best_anchor_phase_node[1]+1) ##because 0 based indexing used here but 1 based in locations

                for ind_no, ses_indX in enumerate([ses_ind,ref_task]):
                    session=non_repeat_ses[ses_indX]

                    
                    location_mat_=np.load(Input_folder+'Location_'+mouse_recday+'_'+str(session)+'.npy')
                    if len(location_mat_)==0:
                        continue
                    occupancy_mat=np.reshape(location_mat_,(num_states,len(location_mat_),\
                                                            len(location_mat_.T)//num_states))
                    occupancy_conc=np.concatenate(location_mat_)

                    phase_mat=np.zeros(np.shape(occupancy_mat))
                    phase_mat[:,:,30:60]=1
                    phase_mat[:,:,60:90]=2
                    phase_conc=np.concatenate(np.hstack(phase_mat))
                    
                    exec('ephys_=ephys_ses_'+str(session)+'_')
                    ephys_neuron_=ephys_[neuron]
                    neuron_mat=ephys_neuron_
                    neuron_conc=np.concatenate(neuron_mat)

                    tone_aligned_activity=neuron_mat


                    mean_=np.mean(tone_aligned_activity,axis=0)
                    sem_=st.sem(tone_aligned_activity,axis=0)
                    mean_smooth=smooth_circular(mean_)


                    timestamps_=np.where((np.logical_and(occupancy_conc==location, phase_conc==phase_)))[0]
                    long_stays=np.where(rank_repeat2(occupancy_conc)>thr_visit)[0]
                    timestamps=np.intersect1d(timestamps_,long_stays-(thr_visit+1))
                    if len(timestamps)>0:

                        timestamps_start=timestamps[(np.hstack((1,np.diff(timestamps)>thr_visit))).astype(bool)]
                        timestamps_end=timestamps[(np.hstack((np.diff(timestamps)>thr_visit,1))).astype(bool)]
                        aligned_activity=np.asarray([neuron_conc[ii:ii+num_bins]\
                                                     if len(neuron_conc[ii:ii+num_bins])==num_bins\
                                                     else np.repeat(np.nan,num_bins) for ii in timestamps_start])

                        mean_=np.nanmean(aligned_activity,axis=0)
                        sem_=st.sem(aligned_activity,axis=0,nan_policy='omit')
                        mean_smooth=smooth_circular(mean_)
                        sem_smooth=smooth_circular(sem_)
                        if len(timestamps_start)==1:
                            sem_smooth=np.repeat(0,num_bins)

                        if np.nanmean(mean_smooth)==0 or np.isnan(np.nanmean(mean_smooth))==True:
                            mean_smooth=np.repeat(np.nan,num_bins)

                    else:
                        mean_smooth=np.repeat(np.nan,num_bins)
                        sem_smooth=np.repeat(np.nan,num_bins)



                    mean_smooth_all[ses_ind,ind_no]=mean_smooth
            except Exception as e:
                if neuron==0:
                    print(e)
                    exc_type, exc_obj, exc_tb = sys.exc_info()
                    fname = os.path.split(exc_tb.tb_frame.f_code.co_filename)[1]
                    print(exc_type, fname, exc_tb.tb_lineno)
                    print('session not analysed')
                continue

        phase_norm_mean=np.tile(np.repeat(np.arange(num_phases),num_bins_state/num_phases),num_states)
        phase_norm_mean_states=np.reshape(phase_norm_mean,(num_states,num_bins_state))
        corr_all=[]
        for ses_ind in np.arange(len(mean_smooth_all)):
            mean_smooth_all_means=[np.asarray([np.nanmean(mean_smooth_all[ses_ind,jj,num_bins_state*ii:\
                                                                          num_bins_state*(ii+1)]\
            [phase_norm_mean_states[ii]==pref_phase]) for ii in range(num_states)]) for jj in np.arange(2)]
            
            if np.isnan(np.mean(mean_smooth_all_means))==False: 
                corr_ses=st.pearsonr(mean_smooth_all_means[0],mean_smooth_all_means[1])[0]
            else:
                corr_ses=np.nan
            corr_all.append(corr_ses)

        corr_mean=np.nanmean(corr_all)
        
        state_corrs_allneurons[neuron]=corr_mean
    Spatial_anchoring_dic['Cross_val_corr'][mouse_recday]=state_corrs_allneurons

In [ ]:
###Trial by trial anchoring analysis

day_type='combined_ABCDonly'
Anchor_trial_dic=rec_dd()

num_trials_thr=5

shifts=np.arange(11)-5 ##this is used for shifting across trials
num_shifts=len(shifts)
use_mean=False
use_timestamps=True

skip_analysed=False
regression=False
use_individualsession_anchor=False
thr_visit=2

for use_individualsession_anchor in [True,False]:
    if use_individualsession_anchor==True:
        name_addition='_cross_val'
    else:
        name_addition=''
    for mouse_recday in np.load(Input_folder+day_type+'_days.npy'):
        print(mouse_recday)
        
        if day_type=='combined_ABCDonly':
            num_states=4

        num_bins=num_states*90
        angle_correction=360/num_bins

        num_neurons=len(np.load(Input_folder+'Neuron_raw_'+mouse_recday+'_0.npy'))

        if skip_analysed==True:
            if len(Anchor_trial_dic['phase_location'][mouse_recday])>0:
                if np.shape(Anchor_trial_dic['phase_location'][mouse_recday])[1]==num_neurons:
                    print('Already analysed')
                    continue

        num_sessions=len(np.load(Input_folder+'awake_session_behaviour_'+mouse_recday+'.npy'))

        ##defining sessions to use
        sessions=np.load(Input_folder+'Task_num_'+mouse_recday+'.npy')
        num_refses=len(np.unique(sessions))
        num_comparisons=num_refses-1
        repeat_ses=np.where(rank_repeat(sessions)>0)[0]
        non_repeat_ses=non_repeat_ses_maker(mouse_recday) 
        
        num_trials=np.load(Input_folder+'Num_trials_'+mouse_recday+'.npy')
        
        trials_completed_ses=np.where(num_trials>2)[0]
        non_repeat_ses=np.intersect1d(non_repeat_ses,trials_completed_ses)
        num_nonrepeat_sessions=len(non_repeat_ses)


        best_shift_all=np.zeros((num_neurons,num_nonrepeat_sessions))
        phase_location_all=np.zeros((num_nonrepeat_sessions,num_neurons,2))
        best_shift_all[:]=np.nan
        phase_location_all[:]=np.nan

        print('Importing Ephys')
        for ses_ind_ind, ses_ind in enumerate(non_repeat_ses):
            print(ses_ind)
            ##### Importing Ephys
            try:
                ephys_ = np.load(Input_folder+'Neuron_'+mouse_recday+'_'+str(ses_ind)+'.npy')
            except:
                print('No Ephys')
                continue
            ##Importing locations
            location_mat_=np.load(Input_folder+'Location_'+mouse_recday+'_'+str(ses_ind)+'.npy')
            if len(location_mat_)==0:
                print('No entries in location file')
                continue
            occupancy_mat=np.reshape(location_mat_,(num_states,len(location_mat_),len(location_mat_.T)//num_states))
            occupancy_conc=np.concatenate(location_mat_)


            phase_mat=np.zeros(np.shape(occupancy_mat))
            phase_mat[:,:,30:60]=1
            phase_mat[:,:,60:90]=2

            phase_conc=np.concatenate(np.hstack(phase_mat))
            occupancy_mat_=location_mat_
            phase_mat_=np.hstack(phase_mat)

            ephys_neuron_=neuron_mat=ephys_[0]

            if len(neuron_mat)==0 or np.shape(occupancy_mat)[1]<num_trials_thr:
                print('not enough trials')
                continue

            if np.shape(occupancy_mat)[1]<num_trials_thr:
                print('not enough trials')
                continue

            tone_aligned_activity=neuron_mat
            min_trials=int(np.min([len(occupancy_mat_),len(tone_aligned_activity)]))

            
            if len(Spatial_anchoring_dic['best_node_phase_used'][mouse_recday])==0:
                print('Not used')
                continue

            for neuron in np.arange(num_neurons):

                if use_individualsession_anchor==True:
                    anchors=(Spatial_anchoring_dic['best_node_phase_used'][mouse_recday][:,:,neuron]).astype(int)
                    phase_=anchors[0,ses_ind_ind]
                    location_=anchors[1,ses_ind_ind]
                    location=location_+1
                    name_addition='_cross_val'
                else:
                    anchor=(Spatial_anchoring_dic['Best_anchor_all'][mouse_recday][neuron]).astype(int)
                    location_=anchor[1]
                    phase_=anchor[0]
                    location=int(location_+1)
                    name_addition=''

                phase_location_all[ses_ind_ind,neuron]=phase_,location_

                ephys_neuron_=ephys_[neuron]
                neuron_mat=ephys_neuron_
                neuron_conc=np.concatenate(neuron_mat)

                ###tone aligned activity
                tone_aligned_activity=neuron_mat
                anchor_mat=(np.logical_and(phase_mat_==phase_,occupancy_mat_==location)).astype(int)
                min_trials=int(np.min([len(anchor_mat),len(tone_aligned_activity)]))

                tone_aligned_activity_matched=tone_aligned_activity[:min_trials]
                anchor_mat_matched=anchor_mat[:min_trials]
                corr_all=np.zeros(num_bins)
                for shift in range(num_bins):
                    tone_aligned_activity_shifted=np.roll(tone_aligned_activity_matched,-shift)#,axis=1)

                    if use_mean==False:
                        corr_mat=np.corrcoef(anchor_mat_matched,tone_aligned_activity_shifted)
                        cross_corr_mat=corr_mat[min_trials:,:min_trials]
                        corr_all[shift]=np.nanmean(np.diagonal(cross_corr_mat))
                    elif use_mean==True:
                        mean_neuron=np.mean(tone_aligned_activity_shifted,axis=0)
                        mean_anchor=np.mean(anchor_mat_matched,axis=0)
                        corr_all[shift]=st.pearsonr(mean_anchor,mean_neuron)[0]

                best_shift_corr=np.argmax(corr_all)

                #######
                if use_timestamps==True:
                    timestamps_=np.where((np.logical_and(occupancy_conc==location, phase_conc==phase_)))[0]
                    long_stays=np.where(rank_repeat2(occupancy_conc)>thr_visit)[0]
                    timestamps=np.intersect1d(timestamps_,long_stays-(thr_visit+1))
                    if len(timestamps)>0:

                        timestamps_start=timestamps[(np.hstack((1,np.diff(timestamps)>thr_visit))).astype(bool)]
                        timestamps_end=timestamps[(np.hstack((np.diff(timestamps)>thr_visit,1))).astype(bool)]
                        aligned_activity=np.asarray([neuron_conc[ii:ii+num_bins]\
                                                     if len(neuron_conc[ii:ii+num_bins])==num_bins\
                                                     else np.repeat(np.nan,num_bins) for ii in timestamps_start])
                        mean_=np.nanmean(aligned_activity,axis=0)
                        sem_=st.sem(aligned_activity,axis=0,nan_policy='omit')
                        mean_smooth=smooth_circular(mean_)

                        best_shift=np.argmax(mean_smooth)

                    else:
                        best_shift=np.nan

                else:
                    best_shift=best_shift_corr

                
                best_shift_all[neuron,ses_ind_ind]=best_shift


        Anchor_trial_dic['phase_location'+name_addition][mouse_recday]=np.asarray(phase_location_all)
        Anchor_trial_dic['Best_shift_time'+name_addition][mouse_recday]=np.asarray(best_shift_all)
      

In [ ]:
####One best shift time per neuron
for mouse_recday in np.load(Input_folder+day_type+'_days.npy'):
    print(mouse_recday)

    num_neurons=len(np.load(Input_folder+'Neuron_raw_'+mouse_recday+'_0.npy'))
    most_common_angle_all=np.zeros(num_neurons)
    most_common_angle_all[:]=np.nan
    
    mean_angle_all=np.zeros(num_neurons)
    mean_angle_all[:]=np.nan
    for neuron in np.arange(num_neurons):
        most_common_anchor_bool=Spatial_anchoring_dic['most_common_anchor_bool'][mouse_recday][neuron]
        if len(most_common_anchor_bool)==0:
            print('Not used')
            continue
        Best_shift_time=Anchor_trial_dic['Best_shift_time'][mouse_recday][neuron]
        Best_shift_time_disc=(Best_shift_time[most_common_anchor_bool]//30)*30
        
        mean_angle=np.rad2deg(st.circmean(np.deg2rad(remove_nan(Best_shift_time[most_common_anchor_bool]))))
        
        if len(remove_nan(Best_shift_time_disc))==0:
            most_common_angle=np.nan
        else:  
            most_common_angle=st.mode(remove_nan(Best_shift_time_disc))[0][0]
      
        mean_angle_all[neuron]=mean_angle
        most_common_angle_all[neuron]=most_common_angle

    Anchor_trial_dic['Best_shift_time_mostcommon'][mouse_recday]=most_common_angle_all
    Anchor_trial_dic['Best_shift_time_mean'][mouse_recday]=mean_angle_all
    Anchor_trial_dic['Best_shift_time_mostcommon_all'][mouse_recday]=Best_shift_time


In [ ]:
###Single anchoring analysis - subsetted by lag from anchor 

condition='non-zero-strict' #'non-zero','non-zero-strict','>N-1states'

if day_type=='combined_ABCDonly':
    num_states=4

num_bins=num_states*90

day_type='combined_ABCDonly'
    
coh_thr1=(360/num_states)//2
coh_thr2=360-coh_thr1


if condition=='non-zero':
    min_thr=(360/num_states)//3
    max_thr=360-min_thr
elif condition=='non-zero-strict':
    min_thr=(360/num_states)
    max_thr=360-min_thr
elif confition=='>N-1states':
    min_thr=(360/num_states)*(num_states-1)
    max_thr=360

for mouse_recday in np.load(Input_folder+day_type+'_days.npy'):


    print(mouse_recday)
    #try:
    num_sessions=len(np.load(Input_folder+'awake_session_behaviour_'+mouse_recday+'.npy'))
    num_neurons=len(np.load(Input_folder+'Neuron_raw_'+mouse_recday+'_0.npy')) 
    print(num_sessions)

    sessions=np.load(Input_folder+'Task_num_'+mouse_recday+'.npy')
 
    repeat_ses=np.where(rank_repeat(sessions)>0)[0]
    non_repeat_ses=non_repeat_ses_maker(mouse_recday)  ###this defines the sessions used
    ###only the first session from each task is used
    
    num_trials=np.load(Input_folder+'Num_trials_'+mouse_recday+'.npy')
    
    trials_completed_ses=np.where(num_trials>2)[0]
    non_repeat_ses=np.intersect1d(non_repeat_ses,trials_completed_ses)

    

    
    num_refses=len(non_repeat_ses)
    num_comparisons=num_refses-1
   
    dists_test=Spatial_anchoring_dic['Dists_all'][mouse_recday]
    angles_test=Spatial_anchoring_dic['Angles_all'][mouse_recday]

    
    if day_type=='combined_ABCDonly':
        phase_tuning=np.load(Input_folder+'Phase_'+mouse_recday+'.npy')
        state_tuning=np.load(Input_folder+'State_'+mouse_recday+'.npy')

        
    neurons_tuned=np.where(state_tuning==True)[0]
    neurons_usedx=Spatial_anchoring_dic['Neuron_used_histogram'][mouse_recday]

    Anchor_lags=Anchor_trial_dic['Best_shift_time'][mouse_recday]    
    Anchor_lags_mean=np.rad2deg(st.circmean(np.deg2rad(Anchor_lags),axis=1,nan_policy='omit'))
    Anchor_trial_dic['Anchor_lags_mean'][mouse_recday]=Anchor_lags_mean

    non_zero_anchored=np.where(np.logical_and(Anchor_lags_mean>min_thr,Anchor_lags_mean<max_thr)==True)[0]
    neurons_used=np.intersect1d(neurons_tuned,non_zero_anchored) ##remove zero lag neurons

    if len(neurons_used)==0:
        print('Not used')
        continue
    
    if day_type=='combined_ABCDonly':
        tuning_state_bool_day=np.load(Input_folder+'tuning_state_boolean_'+mouse_recday+'.npy')
        
        

        
    num_peaks_all=np.vstack(([np.sum(tuning_state_bool_day[ses_ind],axis=1)\
                              for ses_ind in np.arange(len(tuning_state_bool_day))])).T

    dists_used=dists_test[:,neurons_used] 
    angles_used=angles_test[:,neurons_used]
    num_peaks_used=num_peaks_all[neurons_used]

    ###making histograms by averaging across all training-test splits
    angles_spatialanchor_num_all=[] 
    angles_spatialanchor_cohprop_all=[] 
    for ses_ind in np.arange(num_refses): 
        angles_used[ses_ind,num_peaks_used[:,ses_ind]==0]=np.nan
        angles_spatialanchor=remove_nan(angles_used[ses_ind]) 
        if len(angles_spatialanchor)>0: 
            coh_prop=len(np.where(np.logical_or(angles_spatialanchor<coh_thr1 ,angles_spatialanchor>coh_thr2))[0])\
            /len(angles_spatialanchor) 
            angles_spatialanchor_num=np.histogram(angles_spatialanchor,np.linspace(0,num_bins,37))[0] 
        else: 
            coh_prop=np.nan 
            angles_spatialanchor_num=np.repeat(np.nan,36) 

        angles_spatialanchor_cohprop_all.append(coh_prop) 
        angles_spatialanchor_num_all.append(angles_spatialanchor_num) 
    angles_spatialanchor_num_mean=np.nanmean(np.asarray(angles_spatialanchor_num_all),axis=0) 
    angles_spatialanchor_cohprop_mean=np.nanmean(angles_spatialanchor_cohprop_all) 

    polar_plot_stateX(angles_spatialanchor_num_mean,angles_spatialanchor_num_mean, 
                      angles_spatialanchor_num_mean,color='black',labels='angles',plot_type='bar') 
    plt.show() 

    Spatial_anchoring_dic['MeanAngles_spatial_Anchor_best_nonzero'][mouse_recday]=angles_spatialanchor_num_mean 
    Spatial_anchoring_dic['Coherent_proportion_nonzero'][mouse_recday]=angles_spatialanchor_cohprop_mean 
    Spatial_anchoring_dic['Neuron_used_histogram_nonzero'][mouse_recday]=neurons_used


In [ ]:
angles_spatialanchor_num=np.nansum(np.vstack(([Spatial_anchoring_dic['MeanAngles_spatial_Anchor_best_nonzero']\
                                               [mouse_recday]\
            for mouse_recday in np.load(Input_folder+day_type+'_days.npy')
        if len(Spatial_anchoring_dic['MeanAngles_spatial_Anchor_best_nonzero'][mouse_recday])>0])),axis=0)
print(angles_spatialanchor_num) 

polar_plot_stateX(angles_spatialanchor_num,angles_spatialanchor_num, 
                  angles_spatialanchor_num,color='black',labels='angles',plot_type='bar') 
plt.tick_params(axis='both',  labelsize=20)
plt.tick_params(width=2, length=6)
plt.savefig(Output_folder+'/Spatial_anchoring_histogram_nonspatial.svg', bbox_inches = 'tight',\
            pad_inches = 0)
plt.show()

In [ ]:
total_coh=[]
total_neurons=[]
for mouse_recday in np.load(Input_folder+day_type+'_days.npy'):
    print(mouse_recday)
    try:

        total_coh.append(Spatial_anchoring_dic['Coherent_proportion_nonzero'][mouse_recday]*\
        len(Spatial_anchoring_dic['Neuron_used_histogram_nonzero'][mouse_recday]))
        total_neurons.append(len(Spatial_anchoring_dic['Neuron_used_histogram_nonzero'][mouse_recday]))
    except:
        print('Not used')
print('Total coherent proportion: '+str(np.nansum(total_coh)/np.nansum(total_neurons)))

In [ ]:
print(two_proportions_test(np.nansum(total_coh), np.nansum(total_neurons),\
                           np.nansum(total_neurons)*(1/num_states), np.nansum(total_neurons)))
print(np.nansum(total_coh), np.nansum(total_neurons))

In [ ]:
use_tuned=True ###does nothing because already subsetted by phase

mean_bestlag_corr_crossval_nonspatial=[]
for mouse_recday in np.load(Input_folder+day_type+'_days.npy'):
    print(mouse_recday)
    try:
        Neuron_used_histogram_nonzero=Spatial_anchoring_dic['Neuron_used_histogram_nonzero'][mouse_recday]
        Neurons_tuned=np.load(Input_folder+'State_'+mouse_recday+'.npy')
        mean_bestlag_corr_crossval_nonspatial.append(Spatial_anchoring_dic['Cross_val_corr'][mouse_recday]\
                                                     [Neuron_used_histogram_nonzero])
    except:
        print('Not used')


if day_type=='combined_ABCDonly':
    state_tuning=np.hstack(([np.load(Input_folder+'State_zmax_bool_'+mouse_recday+'.npy')\
        for mouse_recday in np.load(Input_folder+day_type+'_days.npy')]))

neurons_tuned=state_tuning
###i.e. phase/state tuned neurons

print('All neurons')

plt.rcParams["figure.figsize"] = (7,5)
plt.rcParams['axes.linewidth'] = 4
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.left'] = True
plt.rcParams['axes.spines.bottom'] = True


mean_bestlag_corr_crossval=remove_nan(np.hstack(([Spatial_anchoring_dic['Cross_val_corr'][mouse_recday]\
                                                  for mouse_recday in np.load(Input_folder+day_type+'_days.npy')]))[neurons_tuned])
    
plt.hist(mean_bestlag_corr_crossval,bins=50,color='grey')
plt.axvline(0,color='black',ls='dashed')
plt.tick_params(axis='both',  labelsize=20)
plt.tick_params(width=2, length=6)
plt.savefig(Output_folder+'SingleAnchor_analysis.svg' , bbox_inches = 'tight', pad_inches = 0)
plt.show()
print(len(mean_bestlag_corr_crossval))
print(st.ttest_1samp(mean_bestlag_corr_crossval,0))
print('')

print('Non spatial neurons')

mean_bestlag_corr_crossval=remove_nan(np.hstack((mean_bestlag_corr_crossval_nonspatial)))

plt.hist(mean_bestlag_corr_crossval,bins=50,color='grey')
plt.axvline(0,color='black',ls='dashed')
plt.tick_params(axis='both',  labelsize=20)
plt.tick_params(width=2, length=6)
plt.savefig(Output_folder+'SingleAnchor_analysis_nonzero.svg' , bbox_inches = 'tight', pad_inches = 0)
plt.show()
print(len(mean_bestlag_corr_crossval))
print(st.ttest_1samp(mean_bestlag_corr_crossval,0))
print('')

In [ ]:
###Plotting lags from anchor
from scipy import stats
for angle_type in ['Best_shift_time_mostcommon','Best_shift_time_mean','Best_shift_time_all']:
    if angle_type=='Best_shift_time_all':
        All_lags=np.hstack(([np.hstack((Anchor_trial_dic['Best_shift_time'][mouse_recday]))\
                     for mouse_recday in np.load(Input_folder+day_type+'_days.npy')]))
    else:
        All_lags=np.hstack(([Anchor_trial_dic[angle_type][mouse_recday] for mouse_recday\
     in np.load(Input_folder+day_type+'_days.npy')]))
        
        
        bool_=np.asarray(np.hstack(([Spatial_anchoring_dic['Anchored_bool'][mouse_recday] for mouse_recday\
                                in np.load(Input_folder+day_type+'_days.npy')])))
        bool__=bool_==True
        
        All_lags=All_lags[bool__]
        All_lags=remove_nan(All_lags)
        
    plt.rcParams['axes.spines.right'] = False
    plt.rcParams['axes.spines.top'] = False
    plt.rcParams['axes.spines.left'] = True
    plt.rcParams['axes.spines.bottom'] = True
    plt.rcParams["figure.figsize"] = (8,6)
    plt.hist(All_lags,bins=12,color='black')
    plt.tick_params(axis='both',  labelsize=20)
    plt.tick_params(width=2, length=6)
    plt.savefig(Output_folder+'Single_Anchor_'+angle_type+'analysis_lags.svg',\
                bbox_inches = 'tight', pad_inches = 0)
    plt.show()
    print(len(All_lags))
    print(stats.kstest(remove_nan(All_lags),stats.uniform.cdf))



In [ ]:
###Saving tuning and anchoring booleans
for mouse_recday in np.load(Input_folder+day_type+'_days.npy'):
    print(mouse_recday)


    phase_bool=np.load(Input_folder+'Phase_'+mouse_recday+'.npy')
    state_bool=np.load(Input_folder+'State_'+mouse_recday+'.npy')
    place_bool=np.load(Input_folder+'Place_'+mouse_recday+'.npy')
    

    
    Anchored_bool=Spatial_anchoring_dic['Anchored_bool'][mouse_recday]
    
    anchor_lag_=Anchor_trial_dic['Anchor_lags_mean'][mouse_recday]
    anchor_dist_=1-np.cos(np.deg2rad(anchor_lag_))
    

    Phase_state_place_anchoring=np.vstack((phase_bool,state_bool,place_bool,Anchored_bool,anchor_dist_)).T
    
    Anchor_trial_dic['Phase_state_place_anchored'][mouse_recday]=Phase_state_place_anchoring
    
    np.save(Input_folder+'Phase_state_place_anchored_'+mouse_recday+'.npy',Phase_state_place_anchoring)
    
    
    ###lagged spatial correlation analysis
    num_neurons=len(np.load(Input_folder+'Neuron_raw_'+mouse_recday+'_0.npy'))
    mean_corrs=np.nanmean(Phase_spatial_corr_dic['corrs_all'][mouse_recday],axis=2)
    thresholds_=Phase_spatial_corr_dic['Threshold'][mouse_recday]

    np.save(Input_folder+'Anchor_lag_'+mouse_recday+'.npy',mean_corrs)
    np.save(Input_folder+'Anchor_lag_threshold_'+mouse_recday+'.npy',thresholds_)
    
##Saving output
for measure in Anchor_trial_dic.keys():
    for mouse_recday in Anchor_trial_dic[measure].keys():
        np.save(Input_folder+'Anchor_trial_'+measure+'_'+mouse_recday+'.npy',\
                Anchor_trial_dic[measure][mouse_recday])
        
for measure in Spatial_anchoring_dic.keys():
    for mouse_recday in Spatial_anchoring_dic[measure].keys():
        np.save(Input_folder+'Spatial_anchoring_'+measure+'_'+mouse_recday+'.npy',\
                Spatial_anchoring_dic[measure][mouse_recday])

In [ ]:
#######################################################
### Figure 6 analysis - predicting upcoming choices ###
#######################################################

In [ ]:
##Can we predict the next step from activity of neurons anchored to different targets? - regression
##note this cell also has the simple FR analysis 

Regression_anchors_dic=rec_dd()
day_type='combined_ABCDonly'

num_phases=3
num_nodes=9
num_states=4

num_trials_tested=10 ##how many trials back to regress out (to control for autocorrelation in behaviour)
num_visits_min=20 ##cutoff for how many visits needed before calculating betas

use_kernel=False
use_anchored_only=True
remove_zero_phase=False
use_all_neurons=False ##if true this uses all neurons in their bump time, not just ones anchored to the current
##placephase - i.e. this is a control 
use_crossval=False
use_distal=False


if day_type=='combined_ABCDonly':
    num_states=4
    state_degrees=360/num_states
    activity_time_options=['bump_time','decision_time','random_time',int(state_degrees),\
                           int(state_degrees*2),int(state_degrees*3)]


num_bins=num_states*90
angle_correction=360/num_bins
if use_distal==True:
    thr_lower=(360/num_states)
else:   
    thr_lower=(360/num_states)//3
thr_upper=num_bins-thr_lower


for num_trials_neural in [0]: #[0,1,5,10]:
    ##how many trials back to take neuronal activity from (to look at attractor properties)
    for activity_time in activity_time_options:
        print('____')
        print(str(activity_time))
        print('')


        for mouse_recday in np.load(Input_folder+day_type+'_days.npy'):
            print(mouse_recday)

            mouse=mouse_recday.split('_',1)[0]
            rec_day=mouse_recday.split('_',1)[1]

            #Importing Ephys
            print('Importing Ephys')
            num_sessions=len(np.load(Input_folder+'awake_session_behaviour_'+mouse_recday+'.npy'))
            num_neurons=len(np.load(Input_folder+'Neuron_raw_'+mouse_recday+'_0.npy'))


            ##Tasks
            Tasks=np.load(Input_folder+'Task_data_'+mouse_recday+'.npy',allow_pickle=True)

            ##defining sessions to use
            sessions=np.load(Input_folder+'Task_num_'+mouse_recday+'.npy')

            num_refses=len(np.unique(sessions))
            num_comparisons=num_refses-1
            repeat_ses=np.where(rank_repeat(sessions)>0)[0]
            non_repeat_ses=non_repeat_ses_maker(mouse_recday)  ###this defines the sessions used
            ###only the first session from each task is used


            
            num_trials_ses=np.load(Input_folder+'Num_trials_'+mouse_recday+'.npy',\
                                        allow_pickle=True)

            trials_completed_ses=np.where(num_trials_ses>2)[0]
            non_repeat_ses=np.intersect1d(non_repeat_ses,trials_completed_ses)
            num_nonrepeat_sessions=len(non_repeat_ses)

            ##Which cells are anchored? 
            Anchored_bool=Spatial_anchoring_dic['Anchored_bool'][mouse_recday]
            Anchored_neurons=np.where(Anchored_bool==True)[0]

            ###which neurons are phase/state tuned?
            if day_type=='combined_ABCDonly':
                tuning_state_bool_day=np.load(Input_folder+'tuning_state_boolean_'+mouse_recday+'.npy')

            num_peaks_all=np.vstack(([np.sum(tuning_state_bool_day[ses_ind],axis=1)\
                                      for ses_ind in np.arange(len(tuning_state_bool_day))])).T


            if day_type=='combined_ABCDonly':
                phase_tuning=np.load(Input_folder+'Phase_'+mouse_recday+'.npy')
                state_tuning=np.load(Input_folder+'State_'+mouse_recday+'.npy')

            neurons_tuned=np.logical_and(state_tuning,phase_tuning)


            ###defining array for regression betas
            if use_kernel==True:
                num_trials_tested_arraysize=1
            else:
                num_trials_tested_arraysize=num_trials_tested

            all_betas_allanchors=np.zeros((num_nonrepeat_sessions,num_phases,num_nodes,\
                                           int(num_trials_tested_arraysize+1)))
            neuron_betas_allanchors=np.zeros((num_nonrepeat_sessions,num_phases,num_nodes))
            norm_FR_nonrepeat=np.zeros((num_nonrepeat_sessions,num_phases,num_nodes,2))
            norm_FR_allcombinations=np.zeros((num_nonrepeat_sessions,num_phases,num_nodes,4))
            mean_activity_ses_neurons=np.zeros((num_nonrepeat_sessions,num_neurons))
            num_anchorvisitsnorm_neurons=np.zeros((num_nonrepeat_sessions,num_neurons))


            all_betas_allanchors[:]=np.nan
            neuron_betas_allanchors[:]=np.nan
            norm_FR_nonrepeat[:]=np.nan
            norm_FR_allcombinations[:]=np.nan
            mean_activity_ses_neurons[:]=np.nan
            num_anchorvisitsnorm_neurons[:]=np.nan

            xy_11_all=[]
            xy_01_all=[]

            ###looping over all sessions (change to used sessions?)
            for ses_ind_ind, ses_ind in enumerate(non_repeat_ses):

                #neurons_tuned_ses=np.where(num_peaks_all[:,ses_ind_ind]>0)[0]
                neurons_tuned_ses=np.where(neurons_tuned==True)[0]

                ##Is the neuron anchored
                Anchored_neurons_1=np.where(Spatial_anchoring_dic['Cross_val_corr'][mouse_recday]>0)[0]
                Anchored_bool=Spatial_anchoring_dic['Anchored_bool_crossval'][mouse_recday][ses_ind_ind]
                Anchored_neurons_2=np.where(Anchored_bool==True)[0]
                Anchored_neurons=Anchored_neurons_2

               
                ##What is the task?
                Task=Tasks[ses_ind]

                ##what is the anchor?
                if use_crossval==True:
                    try:
                        phase_locations_neurons_=((Spatial_anchoring_dic['best_node_phase_used'][mouse_recday]\
                                          [:,ses_ind_ind,:]).astype(int)).T
                    except:
                        print('spatial anchors not found')
                        continue
                    ###what is the lag between the neuron's firing and the anchor? 
                    Best_shift_time_=Anchor_trial_dic['Best_shift_time_cross_val'][mouse_recday]

                else:
                    phase_locations_neurons_=Spatial_anchoring_dic['Best_anchor_all'][mouse_recday]

                    ###what is the lag between the neuron's firing and the anchor? 
                    Best_shift_time_=Anchor_trial_dic['Best_shift_time'][mouse_recday]

                phase_locations_neurons=(np.column_stack((phase_locations_neurons_[:,0],\
                                                         phase_locations_neurons_[:,1]+1))).astype(int)
                if len(phase_locations_neurons)==0:
                    continue
                phase_locations_neurons_unique=np.unique(phase_locations_neurons,axis=0)


                ###defining all non-spatial neurons (i.e. neurons with greater than threshold lag from their anchor)
                nonzero_anchored_neurons=np.where(np.logical_and(Best_shift_time_[:,ses_ind_ind]>thr_lower,\
                                              Best_shift_time_[:,ses_ind_ind]<thr_upper))[0]


                ###defining ephys, occupancy and phase matrices for this session
                try:
                    ephys_=np.load(Input_folder+'Neuron_'+mouse_recday+'_'+str(ses_ind)+'.npy')
                except:
                    print('No Ephys')
                    continue
                if len(ephys_)==0 :
                    print('No Ephys')
                    continue

                if len(ephys_[0])==0:
                    continue


                location_mat_=np.load(Input_folder+'Location_'+mouse_recday+'_'+str(ses_ind)+'.npy')
                occupancy_mat=np.reshape(location_mat_,(num_states,len(location_mat_),\
                                                        len(location_mat_.T)//num_states))
                occupancy_conc=np.concatenate(location_mat_)

                phase_mat=np.zeros(np.shape(occupancy_mat))
                phase_mat[:,:,30:60]=1
                phase_mat[:,:,60:90]=2
                phase_conc=np.concatenate(np.hstack(phase_mat))

                occupancy_mat_=location_mat_
                phase_mat_=np.hstack(phase_mat)

                num_trials=len(occupancy_mat_)


                neurons_used_day=[]
                ###looping over all location-phase conjunctions to find neurons anchored to them and do 
                ##the regression
                for phase_ in np.arange(num_phases):
                    if phase_==0 and remove_zero_phase==True:
                        continue
                    for location_ in np.arange(num_nodes):
                        location=location_+1 ## location arrays are not using zero-based indexing 

                        ##defining place/phase visits
                        placephase_mat=(np.logical_and(phase_mat_==phase_,occupancy_mat_==location)).astype(int)
                        placephase_conc=(np.logical_and(phase_conc==phase_,occupancy_conc==location)).astype(int)
                        visits=np.where(placephase_conc==1)[0]
                        if len(visits)==0:
                            continue

                        ###where are the location/phase conjunctions one step away from the anchor?
                        ##This defines decision times
                        prev_locations_=np.where(mindistance_mat[location_]==1)[0]
                        prev_locations=prev_locations_+1


                        if len(prev_locations)==0:
                            continue

                        prev_phase_=(phase_-1)%3
                        location_conc_prev_=np.max(np.asarray([occupancy_conc==prev_locations[ii] for ii \
                                                     in range(len(prev_locations))]),axis=0)
                        location_conc_prev=location_conc_prev_>0 ##i.e. when visiting location one step from 
                        ##anchor location
                        placephase_conc_prev=(np.logical_and(phase_conc==prev_phase_\
                                                             ,location_conc_prev==True)).astype(int)
                        #i.e. when visiting location one step from anchor location AND phase one step 
                        ##from anchor phase 
                        visits_prev=np.where(placephase_conc_prev==1)[0]
                        if len(visits_prev)==0:
                            continue
                        visits_start_=visits[np.hstack((num_bins,np.diff(visits)))>30] ##only the start of a visits 
                        ##in a given phase
                        visits_prev_end=visits_prev[np.hstack((np.diff(visits_prev),num_bins))>30] 
                        ##only the end of a visit in a given phase


                        visits_start=visits_start_[visits_start_>=num_bins] ###removing first trial
                        visits_prev_end=visits_prev_end[visits_prev_end>num_bins] ###removing first trial
                        visits_prev_end_nodes=occupancy_conc[visits_prev_end] 
                        ##which nodes are visited at decision points

                        if len(visits_prev_end)<num_visits_min:
                            continue

                        ##where does the animal actually go after the decision points i.e when one step away
                        ##from anchor (visits_prev_end) - it can go to anchor or not
                        visited_node_start=np.zeros((len(visits_prev_end),2))
                        visited_node_start[:]=np.nan
                        for visit_prev_ind, visit_prev in enumerate(visits_prev_end):
                            next_90_occ=occupancy_conc[visit_prev+1:visit_prev+91]
                            next_90_phase=phase_conc[visit_prev+1:visit_prev+91]

                            next_node_=next_90_occ[np.logical_and(next_90_occ<10,next_90_occ!=\
                                                                 visits_prev_end_nodes[visit_prev_ind])]
                            ##only taking nodes (not edges) and only nodes that arent the same as the previous nodes
                            ##(because could have moved phases but stayed in same node)

                            ###excluding trials where animal stayed in same location across two phases
                            ###because then cant define decision point
                            if len(next_90_occ)==0 or len(next_node_)==0: 
                                continue
                            next_node=next_node_[0] ###the very first node visited after decision point
                            next_node_phase_start_=np.where(np.logical_and(next_90_phase==phase_,\
                                                                           next_90_occ==next_node))[0]
                            ##when is next node visited at the next phase

                            ##whats the first bin where the next node is visited in the next phase  
                            if len(next_node_phase_start_)==0 or\
                            mindistance_mat[int(visits_prev_end_nodes[visit_prev_ind]-1),int(next_node-1)]!=1:
                                ##note: the second condition is to deal with erroneous tracking where animal position
                                ##jumps more than one node between bins
                                next_node=np.nan
                                next_node_phase_start=np.nan
                            else:
                                next_node_phase_start=next_node_phase_start_[0]

                            visited_node_start[visit_prev_ind]=np.asarray([next_node,\
                                                                           visit_prev+next_node_phase_start])

                        ###we now have visited_node_start which has both the visited node and its timestamp following
                        #each decision point (decision point being when animal was one place and one phase 
                        ##away from anchor)


                        ##Defining dependent variable (location/phase visits)                            
                        location_visited_bool=(visited_node_start[:,0]==location).astype(int) ## 1) DEPENDENT VARIABLE


                        ##i.e. when did animal visit the anchor after each decision point
                        ##the length of this array is all the times where animal was one location and one phase away
                        ##from anchor
                        times_=visited_node_start[:,1] ##times for dependent variable
                        nodes_=visited_node_start[:,0] ##nodes visited for dependent variable

                        ### Did animal visit the anchor location on N previous trials?
                        ## we regress this out to remove any effects of autocorrelation in behaviour 
                        trial_lag_booleans_all=np.zeros((num_trials_tested,len(times_)))
                        trial_lag_booleans_all[:]=np.nan
                        for trial_lag_ in np.arange(num_trials_tested):
                            trial_lag=trial_lag_+1
                            range_tested=[(num_bins*trial_lag)-30,\
                                          (num_bins*trial_lag)+30]
                            ##what range of time lags should we use to look for previous trial visits 
                            ##using num_trials_neural here when predicting behaviour using neural activity from 
                            ##M trials back, this means now the coregressors (animal's previous choices) for the 
                            ##behaviour are lagged by exactly N+M trials back (with tolerance of +/- 30 degrees))


                            ## for each bin in times_ (i.e. each timestamp following each decision point) what (if 
                            ##any) is the bin where animal visited the same location/phase N+M trials back
                            ##Note: for trials less then N+M you will effectively never have visited the same 
                            ##anchor at this trial lag and so the co-regressors for this trial lag will always be 
                            ##zero 
                            visit_at_lag=[np.where(np.logical_and((times_<(times_[ii]-range_tested[0])),\
                                                                  (times_>(times_[ii]-range_tested[1]))))[0]\
                            for ii in range(len(times_))]

                            ###did animal visit the anchor location M+N trials before the current visit time?
                            trial_lag_boolean=np.asarray([np.sum([nodes_[visit_at_lag[ii][jj]]==location\
                                                                  for jj in range(len(visit_at_lag[ii]))])\
                                        if len(visit_at_lag[ii]>0) else 0 for ii in range(len(nodes_))])
                            trial_lag_boolean[trial_lag_boolean>0]=1
                            trial_lag_booleans_all[trial_lag_]=trial_lag_boolean ## 2) CO-REGRESSORS

                            ###Now you have trial_lag_booleans_all which tells you when you visited place/phase 
                            ##anchor exactly N+M trials in the past for each N (and a fixed M) - note M=0 for the 
                            ##main analysis



                        ##find neurons anchored to this location/phase with non-zero distance
                        neurons_anchorednext=np.where(np.logical_and(phase_locations_neurons[:,0]==phase_,\
                                                                phase_locations_neurons[:,1]==location))[0]####
                        neurons_anchorednext_nonzero=np.intersect1d(neurons_anchorednext,\
                                                                    nonzero_anchored_neurons)

                        neurons_anchorednext_nonzero_tuned=np.intersect1d(neurons_anchorednext_nonzero,\
                                                                          neurons_tuned_ses)
                        neurons_anchorednext_nonzero_anchored=np.intersect1d(Anchored_neurons,\
                                                                             neurons_anchorednext_nonzero)
                        neurons_anchorednext_nonzero_anchored_tuned=np.intersect1d(\
                                                                        neurons_anchorednext_nonzero_anchored,\
                                                                                   neurons_tuned_ses)
                        ##i.e. neurons that have the same anchor for half the tasks or more 

                        if use_anchored_only==True:
                            neurons_used=neurons_anchorednext_nonzero_anchored
                        else:
                            neurons_used=neurons_anchorednext_nonzero


                        if use_all_neurons==True:
                            neurons_used=np.arange(num_neurons)
                        ##Below we're getting  mean activity of neurons at different times before anchor visit
                        ##first defining arrays
                        mean_activity_bump_neurons=np.zeros((len(neurons_used),len(location_visited_bool)))

                        mean_activity_bump_neurons[:]=np.nan


                        if len(neurons_used)==0:
                            ##i.e. no neurons anchored to this location/phase
                            all_betas_allanchors[ses_ind_ind,phase_,location_]=np.repeat(np.nan,\
                                                                            num_trials_tested_arraysize+1)
                            neuron_betas_allanchors[ses_ind_ind,phase_,location_]=np.nan
                            continue

                        ##whats the lag for each neuron to its anchor? 
                        
                        best_shift_times=Best_shift_time_[neurons_used,ses_ind_ind]

                        ##mean first trial activity of all neurons
                        mean_allneurons=np.mean(np.mean(ephys_[:,0],axis=1)/np.mean(np.mean(ephys_,axis=1),axis=1))
                        ##normalised for each neuron by mean activity across all trials



                        ###defining primary independent variable (neurons activity at defined time)
                        ###defining time to take neuron's activity (for primary independent variable)
                        neurons_used_day.append(neurons_used)
                        for neuron_ind, neuron in enumerate(neurons_used):
                            ephys_neuron_=ephys_[neuron]
                            neuron_conc=np.hstack((ephys_neuron_))

                            ##defining activity times
                            if activity_time == 'bump_time':
                                gap=num_bins-best_shift_times[neuron_ind]
                                ##i.e. times when neurons should be active on ring attractor
                            elif activity_time == 'decision_time':
                                gap=thr_lower
                                ##i.e. time when animal is about to visit anchor
                            elif activity_time == 'random_time':
                                gap=random.randint(0,num_bins-1)
                            elif isinstance(activity_time,int)==True:
                                gap=((num_bins-best_shift_times[neuron_ind])+int(activity_time))%num_bins
                                ##times 90 degrees shifted from bump time
                            neuron_bump_time_start_=times_-gap ##definitely looking at activity BEFORE anchor visit                               
                            neuron_bump_time_start=(neuron_bump_time_start_).astype(int)


                            ##defining normalised mean activity at selected times (ranging from time to 30 degrees 
                            ##later)
                            mean_activity_bump=np.asarray([np.mean(neuron_conc[neuron_bump_time_start[ii]:\
                                                                               neuron_bump_time_start[ii]+30])\
                                                           for ii in range(len(neuron_bump_time_start))])
                            mean_activity_bump[np.isnan(neuron_bump_time_start_)]=np.nan


                            times_int=(times_).astype(int)
                            mean_activity_trial=np.asarray([np.mean(neuron_conc\
                                                                    [times_int[ii]:times_int[ii]+num_bins])\
                                                            for ii in range(len(times_int))])

                            mean_activity_trial[mean_activity_trial==0]=np.nan ##to avoid dividing by zero
                            mean_activity_bump_neurons[neuron_ind]=mean_activity_bump/mean_activity_trial
                            ##i.e. firing rate as proportion of each neuron's mean firing on a given trial

                            mean_activity_ses=np.nanmean(neuron_conc)
                            mean_activity_ses_neurons[ses_ind_ind,neuron]=mean_activity_ses
                            num_anchorvisitsnorm_neurons[ses_ind_ind,neuron]=len(times_)/num_trials


                        meanofmeans_activity_bump_neurons=\
                        np.nanmean(mean_activity_bump_neurons,axis=0) ##3) INDPENDENT VARIABLE
                        ##mean relative firing rate across ALL neurons anchored to a given place/phase
                        ##i.e. collapsing neuron dimension and just keeping visit dimension


                        ## Final inputs to regression
                        X_=np.column_stack((meanofmeans_activity_bump_neurons,trial_lag_booleans_all.T))
                        y_=location_visited_bool


                        X=X_[~np.isnan(meanofmeans_activity_bump_neurons)]
                        X=np.column_stack((X[:,0]-np.mean(X[:,0]),X[:,1:])) ##de-meaning neuronal activity
                        y=y_[~np.isnan(meanofmeans_activity_bump_neurons)]



                        ###lagging by num_trials_neural
                        times__=times_[~np.isnan(meanofmeans_activity_bump_neurons)]##updated times removing nans
                        if len(times__)==0:
                            continue
                        times_shifted_=times__+num_bins*num_trials_neural
                        indices_trial_lagged_=[[ii,np.where(np.logical_and(times_shifted_[ii]<\
                                                                                     times__+30,times_shifted_[ii]>\
                                                                                     times__-30))[0][0]]\
                                    for ii in range(len(times_shifted_))\
                                    if len(np.where(np.logical_and(times_shifted_[ii]<times__+30,\
                                                                   times_shifted_[ii]>times__-30))[0])>0]
                        if len(indices_trial_lagged_)==0:
                            continue
                        indices_trial_lagged=np.vstack((indices_trial_lagged_))

                        X=np.column_stack((X[indices_trial_lagged[:,0],0],X[indices_trial_lagged[:,1],1:]))
                        y=y[indices_trial_lagged[:,1]]

                        prev0_bool=X[:,2]==0
                        prev1_bool=X[:,2]==1

                        if len(X)==0:
                            continue

                        if use_kernel==True:
                            behaviour_coefficients=np.load(Input_folder+\
                                                           '_previous_chocies_coefficients_ABCD.npy')
                            xdata=np.arange(len(behaviour_coefficients))
                            ydata=behaviour_coefficients
                            popt, pcov = curve_fit(func_decay, xdata, ydata)

                            xdata_new=np.arange(num_trials_tested)
                            y_pred=func_decay(xdata_new, *popt)
                            X_beh=np.mean(X[:,1:]*y_pred,axis=1)
                            X=np.column_stack((X[:,0],X_beh))

                            y=y[~np.isnan(np.mean(X,axis=1))]

                        ##doing the regression
                        if np.sum(abs(np.diff(y)))==0 or len(X)==0: ### i.e. always visited (or always didnt visit) 
                            ##place/phase from all decision points
                            beta_all=np.repeat(np.nan,num_trials_tested_arraysize+1)
                            beta_neurons=np.nan
                        else:
                            clf = LogisticRegression(solver='saga',penalty=None).fit(X, y) 
                            ##,max_iter=10000
                            beta_all=clf.coef_[0]
                            beta_neurons=clf.coef_[0][0]

                        all_betas_allanchors[ses_ind_ind,phase_,location_]=beta_all
                        neuron_betas_allanchors[ses_ind_ind,phase_,location_]=beta_neurons



                        ###simple analysis to check for policy effect - do anchored neurons fire more in trial 
                        ##before animal goes to anchor even when controlling for previous choice
                        X=X_[~np.isnan(meanofmeans_activity_bump_neurons)]
                        X=np.column_stack((X[indices_trial_lagged[:,0],0],X[indices_trial_lagged[:,1],1:]))
                        non_repeat_visits=X[:,1]!=y
                        X_nonrepeat=X[non_repeat_visits,0]
                        y_nonrepeat=y[non_repeat_visits]


                        mean_rates_01=np.mean(X_nonrepeat[y_nonrepeat==1])
                        mean_rates_10=np.mean(X_nonrepeat[y_nonrepeat==0])

                        repeat_visits=X[:,1]==y
                        X_repeat=X[repeat_visits,0]
                        y_repeat=y[repeat_visits]
                        mean_rates_11=np.mean(X_repeat[y_repeat==1])
                        mean_rates_00=np.mean(X_repeat[y_repeat==0])

                        norm_FR_nonrepeat[ses_ind_ind,phase_,location_]=mean_rates_01,mean_rates_10
                        norm_FR_allcombinations[ses_ind_ind,phase_,location_]=\
                        mean_rates_00,mean_rates_01,mean_rates_10,mean_rates_11


                        prev_nodes_=occupancy_conc[visits_prev_end]
                        prev_nodes=prev_nodes_[~np.isnan(meanofmeans_activity_bump_neurons)]
                        ##updated pre nodes removing nans




            if len(xy_11_all)>0:
                xy_11_all=np.vstack((xy_11_all))
            if len(xy_01_all)>0:
                xy_01_all=np.vstack((xy_01_all))

            Regression_anchors_dic[str(activity_time)][num_trials_neural]['All_betas'][mouse_recday]=\
            all_betas_allanchors
            Regression_anchors_dic[str(activity_time)][num_trials_neural]['neuron_betas'][mouse_recday]=\
            neuron_betas_allanchors

            Regression_anchors_dic[str(activity_time)][num_trials_neural]['norm_FR_nonrepeat'][mouse_recday]=\
            norm_FR_nonrepeat
            Regression_anchors_dic[str(activity_time)][num_trials_neural]['norm_FR_allcombinations']\
            [mouse_recday]=norm_FR_allcombinations

            Regression_anchors_dic[str(activity_time)][num_trials_neural]['pr_vs_norm_FR_11'][mouse_recday]=\
            xy_11_all
            Regression_anchors_dic[str(activity_time)][num_trials_neural]['pr_vs_norm_FR_01'][mouse_recday]=\
            xy_01_all


            if len(neurons_used_day)>0:
                num_neurons_used_day=len(np.unique(np.hstack((neurons_used_day))))
            else:
                num_neurons_used_day=0

            Regression_anchors_dic[str(activity_time)][num_trials_neural]['num_neurons_used'][mouse_recday]=\
            num_neurons_used_day

            if activity_time=='bump_time':
                Regression_anchors_dic[num_trials_neural]['mean_activity'][mouse_recday]=mean_activity_ses_neurons
                Regression_anchors_dic[num_trials_neural]['num_anchor_visits'][mouse_recday]=\
                num_anchorvisitsnorm_neurons


In [ ]:
###Firing rates 
all_days_used=np.asarray(list(Regression_anchors_dic[num_trials_neural]['mean_activity'].keys()))
specific_days=np.intersect1d(np.load(Input_folder+day_type+'_days.npy'),all_days_used)



activity_time='bump_time'
num_trials_neural=0
#unconcatenated_FRs_=dict_to_array(Regression_anchors_dic[str(activity_time)][num_trials_neural]['norm_FR_allcombinations'])
unconcatenated_FRs_=np.asarray([Regression_anchors_dic[str(activity_time)][num_trials_neural]\
                                ['norm_FR_allcombinations'][mouse_recday]\
for mouse_recday in specific_days])

print('Anchors as ns')

concatenated_FRs_=[np.vstack((np.hstack((unconcatenated_FRs_[ii])))) for ii in np.arange(len(unconcatenated_FRs_))]
concatenated_FRs_all=np.vstack((concatenated_FRs_)).T

bar_plotX(concatenated_FRs_all,'none',0,2.5,'nopoints','unpaired',0.025)
plt.show()

plt.rcParams['axes.linewidth'] = 4
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False
bar_plotX(concatenated_FRs_all[:2],'none',0,2.5,'nopoints','paired',0.025)
plt.tick_params(axis='both',  labelsize=20)
plt.tick_params(width=2, length=6)

plt.savefig(Output_folder+'/Rate_changedpolicy_0start_peranchor.svg',\
            bbox_inches = 'tight', pad_inches = 0)
plt.show()

bar_plotX(concatenated_FRs_all[2:],'none',0,2.5,'nopoints','paired',0.025)
plt.tick_params(axis='both',  labelsize=20)
plt.tick_params(width=2, length=6)

plt.savefig(Output_folder+'/Rate_changedpolicy_1start_peranchor.svg',\
            bbox_inches = 'tight', pad_inches = 0)
plt.show()

concatenated_FRs_start0_clean=column_stack_clean(concatenated_FRs_all[0],concatenated_FRs_all[1]).T
concatenated_FRs_start1_clean=column_stack_clean(concatenated_FRs_all[2],concatenated_FRs_all[3]).T
#print(st.wilcoxon(concatenated_FRs_start0_clean[0],concatenated_FRs_start0_clean[1]))
#print(st.wilcoxon(concatenated_FRs_start1_clean[0],concatenated_FRs_start1_clean[1]))

wilc0_peranchor=st.wilcoxon(concatenated_FRs_start0_clean[0],concatenated_FRs_start0_clean[1])
wilc1_peranchor=st.wilcoxon(concatenated_FRs_start1_clean[0],concatenated_FRs_start1_clean[1])

N1=len(concatenated_FRs_start0_clean[0])
N2=len(concatenated_FRs_start1_clean[0])
min_N_peranchor=np.min([N1,N2])

print('')
print('Sessions as ns')
concatenated_FRs_mean=np.vstack(([np.nanmean(np.nanmean(unconcatenated_FRs_[ii],axis=1),axis=1)\
                                  for ii in range(len(unconcatenated_FRs_))])).T


#plt.rcParams["figure.figsize"] = (3,6)
plt.rcParams['axes.linewidth'] = 4
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False
bar_plotX(concatenated_FRs_mean,'none',0,3.0,'nopoints','paired',0.025)
plt.tick_params(axis='both',  labelsize=20)
plt.tick_params(width=2, length=6)

plt.savefig(Output_folder+'/Rate_changedpolicy.svg',\
            bbox_inches = 'tight', pad_inches = 0)
plt.show()

#plt.rcParams["figure.figsize"] = (3,6)
plt.rcParams['axes.linewidth'] = 4
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False
bar_plotX(concatenated_FRs_mean[:2],'none',0,3.0,'nopoints','paired',0.025)
plt.tick_params(axis='both',  labelsize=20)
plt.tick_params(width=2, length=6)

plt.savefig(Output_folder+'/Rate_changedpolicy_0start.svg',\
            bbox_inches = 'tight', pad_inches = 0)
plt.show()

#plt.rcParams["figure.figsize"] = (3,6)

bar_plotX(concatenated_FRs_mean[2:],'none',0,3.0,'nopoints','paired',0.025)
plt.tick_params(axis='both',  labelsize=20)
plt.tick_params(width=2, length=6)

plt.savefig(Output_folder+'/Rate_changedpolicy_1start.svg',\
            bbox_inches = 'tight', pad_inches = 0)
plt.show()

concatenated_FRs_mean_start0_clean=column_stack_clean(concatenated_FRs_mean[0],concatenated_FRs_mean[1]).T
concatenated_FRs_mean_start1_clean=column_stack_clean(concatenated_FRs_mean[2],concatenated_FRs_mean[3]).T
wilc0=st.wilcoxon(concatenated_FRs_mean_start0_clean[0],concatenated_FRs_mean_start0_clean[1])
wilc1=st.wilcoxon(concatenated_FRs_mean_start1_clean[0],concatenated_FRs_mean_start1_clean[1])
#print(wilc0)
#print(wilc1)
N1=len(concatenated_FRs_mean_start0_clean[0])
N2=len(concatenated_FRs_mean_start1_clean[0])
min_N=np.min([N1,N2])



plot_scatter(concatenated_FRs_mean_start0_clean[0],concatenated_FRs_mean_start0_clean[1],'none')
plot_scatter(concatenated_FRs_mean_start1_clean[0],concatenated_FRs_mean_start1_clean[1],'none')




concatenated_FRs_mean_day=np.vstack(([np.nanmean(np.nanmean(np.nanmean(unconcatenated_FRs_[ii],axis=1),axis=1),axis=0)\
            for ii in range(len(unconcatenated_FRs_))])).T
print('')
print('Days as ns')
bar_plotX(concatenated_FRs_mean_day,'none',0,3.5,'points','paired',0.025)
plt.savefig(Output_folder+'/Rate_changedpolicy_daysNs.svg',\
            bbox_inches = 'tight', pad_inches = 0)
plt.show()

concatenated_FRs_day_start0_clean=column_stack_clean(concatenated_FRs_mean_day[0],concatenated_FRs_mean_day[1]).T
concatenated_FRs_day_start1_clean=column_stack_clean(concatenated_FRs_mean_day[2],concatenated_FRs_mean_day[3]).T
print(st.wilcoxon(concatenated_FRs_day_start0_clean[0],concatenated_FRs_day_start0_clean[1]))
print(st.wilcoxon(concatenated_FRs_day_start1_clean[0],concatenated_FRs_day_start1_clean[1]))

In [ ]:
for group_, array in {'0start':concatenated_FRs_mean[:2],'1start':concatenated_FRs_mean[2:]}.items():


    data=array[1]-array[0]
    # Filter data using np.isnan
        
    array=array[:,data<(np.nanmean(data)+np.nanstd(data)*5)]
    
    
    plt.rcParams['axes.spines.right'] = True
    plt.rcParams['axes.spines.top'] = True
    plt.rcParams['axes.spines.bottom'] = True
    plt.rcParams['axes.spines.left'] = True
    xy=column_stack_clean(array[0],array[1])
    noplot_scatter(xy[:,0],xy[:,1],color='black')
    plt.tick_params(
        axis='x',          # changes apply to the x-axis
        which='both',      # both major and minor ticks are affected
        bottom=True,      # ticks along the bottom edge are off
        top=False,         # ticks along the top edge are off
        labelbottom=True) # labels along the bottom edge are off
    plt.tick_params(axis='both',  labelsize=20)
    plt.tick_params(width=2, length=6)
    plt.savefig(Output_folder+'/Rate_changedpolicy_'+group_+'_scatter.svg',\
            bbox_inches = 'tight', pad_inches = 0)
    plt.show()

In [ ]:
for group_, array in {'0start':concatenated_FRs_all[:2],'1start':concatenated_FRs_all[2:]}.items():


    data=array[1]-array[0]
    # Filter data using np.isnan
        
    array=array[:,data<(np.nanmean(data)+np.nanstd(data)*5)]
    
    
    plt.rcParams['axes.spines.right'] = True
    plt.rcParams['axes.spines.top'] = True
    plt.rcParams['axes.spines.bottom'] = True
    plt.rcParams['axes.spines.left'] = True
    xy=column_stack_clean(array[0],array[1])
    noplot_scatter(xy[:,0],xy[:,1],color='black')
    plt.tick_params(
        axis='x',          # changes apply to the x-axis
        which='both',      # both major and minor ticks are affected
        bottom=True,      # ticks along the bottom edge are off
        top=False,         # ticks along the top edge are off
        labelbottom=True) # labels along the bottom edge are off
    plt.tick_params(axis='both',  labelsize=20)
    plt.tick_params(width=2, length=6)
    plt.savefig(Output_folder+'/Rate_changedpolicy_'+group_+'_all_scatter.svg',\
            bbox_inches = 'tight', pad_inches = 0)
    plt.show()

In [ ]:
from statsmodels.stats.anova import AnovaRM
  
# Create the data

sessions=np.tile(np.arange(len(concatenated_FRs_mean.T)),len(concatenated_FRs_mean))
Past=np.repeat([0,1], len(concatenated_FRs_mean.T)*2)
Future=np.tile(np.repeat([0,1], len(concatenated_FRs_mean.T)),2)
FR_norm=np.hstack((concatenated_FRs_mean))
dataframe = pd.DataFrame({'sessions': sessions,
                          'Past': Past,
                          'Future': Future,\
                         'FR_norm':FR_norm})

import pingouin as pg

# Compute the 2-way repeated measures ANOVA. This will return a dataframe.
pg.rm_anova(dv='FR_norm', within=['Past','Future'], subject='sessions', data=dataframe)

# Optional post-hoc tests
#pg.pairwise_ttests(dv='FR_norm', within=['Past','Future'], subject='sessions', data=dataframe)

anova_result=dataframe.rm_anova(dv='FR_norm', within=['Past','Future'], subject='sessions')

print(anova_result)



In [ ]:
##Printing results for text
print('Anchor not visited in trial N: n='+str(len(concatenated_FRs_mean_start0_clean[0]))+' tasks, statistic='\
      +str(int(wilc0[0]))+', P='+str(round(wilc0[1],3))+'.')
print('Anchor visited in trial N: n='+str(len(concatenated_FRs_mean_start1_clean[0]))+' tasks, statistic='\
      +str(int(wilc1[0]))+', P='+str(round(wilc1[1],3))+'.')

print('In addition, an ANOVA on all data (N='+str(min_N)+' tasks) showed')
for var_ind, var in enumerate(['Past','Future','Past * Future']):
    F=anova_result['F'][anova_result['Source']==var][var_ind]
    P=anova_result['p-GG-corr'][anova_result['Source']==var][var_ind]
    df1=anova_result['ddof1'][anova_result['Source']==var][var_ind]
    df2=anova_result['ddof2'][anova_result['Source']==var][var_ind]
    
    if P<0.05:
        word='a'
    elif P>0.05 and P<0.1:
        word='a trend towards a'
    else:
        word='no'
        
    if '*' in var:
        var=var.replace('*', 'x')
        main_=' '
        addition=' interaction'
    else:
        main_=' main effect of '
        addition=''
        
    print(word+main_+var+addition+': F='+str(round(F,2))+', P='+str(round(P,3))+', df1='+str(df1)+', df2='+str(df2)+', ')


In [ ]:
##using anchors as ns
anchors=np.tile(np.arange(len(concatenated_FRs_all.T)),len(concatenated_FRs_all))
Past=np.repeat([0,1], len(concatenated_FRs_all.T)*2)
Future=np.tile(np.repeat([0,1], len(concatenated_FRs_all.T)),2)
FR_norm=np.hstack((concatenated_FRs_all))
dataframe = pd.DataFrame({'anchors': anchors,
                          'Past': Past,
                          'Future': Future,\
                         'FR_norm':FR_norm})

import pingouin as pg

# Compute the 2-way repeated measures ANOVA. This will return a dataframe.
pg.rm_anova(dv='FR_norm', within=['Past','Future'], subject='anchors', data=dataframe)

# Optional post-hoc tests
#pg.pairwise_ttests(dv='FR_norm', within=['Past','Future'], subject='sessions', data=dataframe)

dataframe.rm_anova(dv='FR_norm', within=['Past','Future'], subject='anchors')

anova_result=dataframe.rm_anova(dv='FR_norm', within=['Past','Future'], subject='anchors')

print(anova_result)

In [ ]:
##Printing results for text
print('Anchor not visited in trial N: n='+str(len(concatenated_FRs_start0_clean[0]))+' anchors, statistic='\
      +str(int(wilc0_peranchor[0]))+', P='+str(round(wilc0_peranchor[1],3))+'.')
print('Anchor visited in trial N: n='+str(len(concatenated_FRs_start1_clean[0]))+' anchors, statistic='\
      +str(int(wilc1_peranchor[0]))+', P='+str(round(wilc1_peranchor[1],3))+'.')

if round(wilc1_peranchor[1],3)==0:
    print(wilc1_peranchor[1])
          
if round(wilc0_peranchor[1],3)==0:
    print(wilc0_peranchor[1])


print('In addition, an ANOVA on all data (N='+str(min_N_peranchor)+' anchors) showed')
for var_ind, var in enumerate(['Past','Future','Past * Future']):
    F=anova_result['F'][anova_result['Source']==var][var_ind]
    P=anova_result['p-GG-corr'][anova_result['Source']==var][var_ind]
    df1=anova_result['ddof1'][anova_result['Source']==var][var_ind]
    df2=anova_result['ddof2'][anova_result['Source']==var][var_ind]
    
    if P<0.05:
        word='a'
    elif P>0.05 and P<0.1:
        word='a trend towards a'
    else:
        word='no'
        
    if '*' in var:
        var=var.replace('*', 'x')
        main_=' '
        addition=' interaction'
    else:
        main_=' main effect of '
        addition=''
        
    print(word+main_+var+addition+': F='+str(round(F,2))+', P='+str(round(P,3))+', df1='+str(df1)+', df2='+str(df2)+', ')


In [ ]:
all_days_used=np.asarray(list(Regression_anchors_dic[num_trials_neural]['mean_activity'].keys()))
specific_days=np.intersect1d(np.load(Input_folder+day_type+'_days.npy'),all_days_used)

all_betas_allconditions=[]
mean_betas_allconditions=[]
num_trials_neural=0
for activity_time in activity_time_options:
    #print(str(activity_time))
    #all_betas_=dict_to_array(Regression_anchors_dic[str(activity_time)][num_trials_neural]['neuron_betas'])
    all_betas_=np.asarray([Regression_anchors_dic[str(activity_time)][num_trials_neural]['neuron_betas'][mouse_recday]\
    for mouse_recday in specific_days])
    all_betas=remove_nan(concatenate_complex2(np.concatenate(concatenate_complex2(all_betas_))))

    all_betas_allconditions.append(all_betas)
    #plt.hist(all_betas,bins=np.linspace(-3,3,60))
    #plt.axvline(0,color='black',ls='dashed')
    #plt.show()
    #print(st.ttest_1samp(all_betas,0))
    
    mean_betas=np.nanmean(np.vstack(((np.vstack((all_betas_)).T))).T,axis=1)
    tt_test_=st.ttest_1samp(remove_nan(mean_betas),0)
    activity_time__=np.copy(activity_time)

    if isinstance(activity_time, int)==True:
        activity_time_str=str(activity_time)+' degree shifted time'
    else:
        activity_time_str=activity_time

    activity_time_nounderscore=activity_time_str.replace('_', ' ')
    print('"'+str(activity_time_nounderscore)+'": N='+str(len(remove_nan(mean_betas)))+' tasks, statistic='+\
          str(round(tt_test_[0],2))+', P='+str(round(tt_test_[1],3))+', df='+str(tt_test_.df)+'; ')
    if round(tt_test_[1],3)==0:
        print(tt_test_[1])
     #'''“bump time”: N=126 sessions, statistic=2.68, P=0.008, df=125'''
    mean_betas_allconditions.append(mean_betas)

#all_betas_allconditions=np.asarray(all_betas_allconditions)
#bar_plotX(all_betas_allconditions.T,'none',-0.3,0.3,'nopoints','unpaired',0.025)
plt.rcParams['axes.linewidth'] = 4
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False
                
mean_betas_allconditions=np.asarray(mean_betas_allconditions)
bar_plotX(mean_betas_allconditions,'none',-0.4,0.4,'nopoints','unpaired',0.025)
plt.tick_params(axis='both',  labelsize=20)
plt.tick_params(width=2, length=6)
plt.savefig(Output_folder+'/Neuron_Behaviour_regression.svg', bbox_inches = 'tight', pad_inches = 0)

In [ ]:
means=np.nanmean(mean_betas_allconditions,axis=1)
sems=st.sem(mean_betas_allconditions,axis=1)

plt.rcParams['axes.linewidth'] = 4
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.bottom'] = False

data=mean_betas_allconditions.T
# Filter data using np.isnan
mask = ~np.isnan(data)
filtered_data = [d[m] for d, m in zip(data.T, mask.T)]

sns.swarmplot(filtered_data, color='white',edgecolor='black',linewidth=1)
plt.axhline(0,ls='dashed',color='black')
plt.tick_params(axis='both',  labelsize=20)
plt.tick_params(width=2, length=6)
plt.tick_params(
    axis='x',          # changes apply to the x-axis
    which='both',      # both major and minor ticks are affected
    bottom=False,      # ticks along the bottom edge are off
    top=False,         # ticks along the top edge are off
    labelbottom=False) # labels along the bottom edge are off
plt.savefig(Output_folder+'/Neuron_Behaviour_regression_swarm.svg', bbox_inches = 'tight', pad_inches = 0)
plt.show()